In [ ]:
# ==========================================
# RESULT STORAGE FOR FINAL SUMMARY
# ==========================================

# These dictionaries collect results from the method
# sections so the final tables are built automatically.

master_result_store = {}

def save_master_result(method, mechanism, beta1, se, ci_lower, ci_upper, bias):
    master_result_store[(method, mechanism)] = {
        "Method": method,
        "Mechanism": mechanism,
        "Beta1": beta1,
        "SE": se,
        "CI_Lower": ci_lower,
        "CI_Upper": ci_upper,
        "Bias": bias
    }

point_estimate_store = {}

def save_point_estimates(method, mechanism, params):
    point_estimate_store[(method, mechanism)] = {
        "Method": method,
        "Mechanism": mechanism,
        "Beta0": params["const"],
        "Beta1": params["X1"],
        "Beta2": params["X2"],
        "Beta3": params["X3"]
    }


In [530]:
# ==========================================
# STEP 1: CREATE COMPLETE MULTIVARIABLE DATA
# ==========================================

import numpy as np
import pandas as pd
import statsmodels.api as sm

from scipy.special import expit

from scipy.stats import t

# Reproducibility
np.random.seed(42)

# Number of observations
n = 100


# ------------------------------------------
# Generate three predictors
# ------------------------------------------

X1 = np.random.normal(
    loc=0,
    scale=1,
    size=n
)

X2 = np.random.normal(
    loc=0,
    scale=1,
    size=n
)

X3 = np.random.normal(
    loc=0,
    scale=1,
    size=n
)


# ------------------------------------------
# Generate random error
# ------------------------------------------

epsilon = np.random.normal(
    loc=0,
    scale=1,
    size=n
)


# ------------------------------------------
# Generate Y
#
# True model:
# Y = 5 + 2X1 - 1.5X2 + X3 + epsilon
# ------------------------------------------

Y = (
    5
    + 2 * X1
    - 1.5 * X2
    + X3
    + epsilon
)


# ------------------------------------------
# Put everything into one DataFrame
# ------------------------------------------

df = pd.DataFrame({
    "X1": X1,
    "X2": X2,
    "X3": X3,
    "Y": Y
})


print(df.head())

print("\nShape of dataset:", df.shape)

         X1        X2        X3          Y
0  0.496714 -1.415371  0.357787   7.645277
1 -0.138264 -0.420645  0.560785   5.355043
2  0.647689 -0.342715  1.083051   8.639794
3  1.523030 -0.802277  1.053802  10.913648
4 -0.234153 -0.161286 -1.377669   3.375051

Shape of dataset: (100, 4)


In [531]:
# ==========================================
# STEP 2: BASELINE REGRESSION
# ==========================================

# True coefficients
beta0_true = 5
beta1_true = 2
beta2_true = -1.5
beta3_true = 1


# Predictor matrix
X_model = sm.add_constant(
    df[["X1", "X2", "X3"]]
)


# Create OLS model
model = sm.OLS(
    df["Y"],
    X_model
)


# Fit model
results = model.fit()


# Show regression summary
print(results.summary())


# Extract estimated coefficients
beta_hat = results.params

print("\nEstimated coefficients:")
print(beta_hat)

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.910
Model:                            OLS   Adj. R-squared:                  0.907
Method:                 Least Squares   F-statistic:                     322.1
Date:                Fri, 25 Sep 2026   Prob (F-statistic):           5.85e-50
Time:                        12:25:52   Log-Likelihood:                -127.46
No. Observations:                 100   AIC:                             262.9
Df Residuals:                      96   BIC:                             273.3
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          5.0875      0.089     57.012      0.0

In [532]:
# ==========================================
# STEP 3: CREATE MCAR
# ==========================================

# Same 20% probability for everybody
missing_prob_mcar = 0.20


# Generate one random number for each row
random_values_mcar = np.random.rand(n)


# True = make Y missing
missing_indicator_mcar = (
    random_values_mcar < missing_prob_mcar
)


# Start from the COMPLETE dataset
df_mcar = df.copy()


# Replace selected Y values with NaN
df_mcar.loc[
    missing_indicator_mcar,
    "Y"
] = np.nan


# Check missingness
print("\n===== MCAR =====")

print(
    "Missing Y values:",
    df_mcar["Y"].isna().sum()
)

print(
    "Missing percentage:",
    df_mcar["Y"].isna().mean() * 100
)


===== MCAR =====
Missing Y values: 25
Missing percentage: 25.0


In [533]:
# ==========================================
# STEP 4: CREATE MAR
# ==========================================

# If X1 < 0:
#     30% probability of Y being missing
#
# If X1 >= 0:
#     10% probability of Y being missing

mar_prob = np.where(
    df["X1"] < 0,
    0.30,
    0.10
)


# New random draw for the MAR experiment
random_values_mar = np.random.rand(n)


# Decide which observations become missing
missing_indicator_mar = (
    random_values_mar < mar_prob
)


# Start again from COMPLETE data
df_mar = df.copy()


# Make selected Y values missing
df_mar.loc[
    missing_indicator_mar,
    "Y"
] = np.nan


# Check missingness
print("\n===== MAR =====")

print(
    "Missing Y values:",
    df_mar["Y"].isna().sum()
)

print(
    "Missing percentage:",
    df_mar["Y"].isna().mean() * 100
)


# Verify that missingness is related to X1

mean_x1_observed = (
    df_mar.loc[
        df_mar["Y"].notna(),
        "X1"
    ].mean()
)

mean_x1_missing = (
    df_mar.loc[
        df_mar["Y"].isna(),
        "X1"
    ].mean()
)

print(
    "Mean X1 when Y is observed:",
    mean_x1_observed
)

print(
    "Mean X1 when Y is missing:",
    mean_x1_missing
)



===== MAR =====
Missing Y values: 25
Missing percentage: 25.0
Mean X1 when Y is observed: 0.05522519848762293
Mean X1 when Y is missing: -0.5810616650392442


In [534]:
# ==========================================
# STEP 5: CREATE MNAR
# ==========================================

# Standardize Y first so that the
# missingness mechanism is not too extreme

Y_standardized = (
    (df["Y"] - df["Y"].mean())
    / df["Y"].std()
)


# Probability of missingness depends
# directly on Y
mnar_prob = expit(
    -1.3863 + 0.5 * Y_standardized
)


# New random draw for MNAR
random_values_mnar = np.random.rand(n)


# Decide which observations become missing
missing_indicator_mnar = (
    random_values_mnar < mnar_prob
)


# Start from COMPLETE data
df_mnar = df.copy()


# Hide selected Y values
df_mnar.loc[
    missing_indicator_mnar,
    "Y"
] = np.nan


# Check missingness
print("\n===== MNAR =====")

print(
    "Missing Y values:",
    df_mnar["Y"].isna().sum()
)

print(
    "Missing percentage:",
    df_mnar["Y"].isna().mean() * 100
)


# Verify using the ORIGINAL complete df
# because df_mnar no longer contains
# the missing Y values

mean_y_missing = (
    df.loc[
        missing_indicator_mnar,
        "Y"
    ].mean()
)

mean_y_observed = (
    df.loc[
        ~missing_indicator_mnar,
        "Y"
    ].mean()
)

print(
    "Mean original Y when Y became missing:",
    mean_y_missing
)

print(
    "Mean original Y when Y remained observed:",
    mean_y_observed
)


===== MNAR =====
Missing Y values: 16
Missing percentage: 16.0
Mean original Y when Y became missing: 5.156512754374756
Mean original Y when Y remained observed: 4.887552878255617


In [ ]:
# ==========================================
# STEP 6A: MCAR + COMPLETE CASE
# ==========================================

df_mcar_cc = df_mcar.dropna()


X_mcar_cc = sm.add_constant(
    df_mcar_cc[["X1", "X2", "X3"]]
)


results_mcar_cc = sm.OLS(
    df_mcar_cc["Y"],
    X_mcar_cc
).fit()


print("\n===== MCAR + COMPLETE CASE =====")

print(
    "Observations:",
    len(df_mcar_cc)
)

print(
    "Estimated coefficients:"
)

print(results_mcar_cc.params)
save_point_estimates("Complete Case", "MCAR", results_mcar_cc.params)


In [536]:
# ============================================
# MCAR: COMPLETE CASE + CORRECTED BOOTSTRAP
# ============================================

df = df_mcar.copy()

# Original complete-case estimate
complete_cases = df.dropna(
    subset=["X1", "X2", "X3", "Y"]
).reset_index(drop=True)

X_original = sm.add_constant(
    complete_cases[["X1", "X2", "X3"]]
)

model_original = sm.OLS(
    complete_cases["Y"],
    X_original
).fit()

beta1_cc = model_original.params["X1"]

# --------------------------------------------
# Bootstrap
# --------------------------------------------

B = 500
bootstrap_beta1 = []

np.random.seed(123)

for b in range(B):

    # Resample from ALL 100 original rows
    sample_indices = np.random.choice(
        len(df),
        size=len(df),
        replace=True
    )

    bootstrap_sample = df.iloc[
        sample_indices
    ].reset_index(drop=True)

    # Apply complete-case analysis INSIDE
    # the bootstrap sample
    bootstrap_complete = bootstrap_sample.dropna(
        subset=["X1", "X2", "X3", "Y"]
    )

    # Skip rare bootstrap samples with
    # too few complete observations
    if len(bootstrap_complete) <= 4:
        continue

    X_boot = sm.add_constant(
        bootstrap_complete[["X1", "X2", "X3"]]
    )

    model_boot = sm.OLS(
        bootstrap_complete["Y"],
        X_boot
    ).fit()

    bootstrap_beta1.append(
        model_boot.params["X1"]
    )

bootstrap_beta1 = np.array(bootstrap_beta1)

# Bootstrap inference
se_cc = np.std(
    bootstrap_beta1,
    ddof=1
)

ci_lower = np.percentile(
    bootstrap_beta1,
    2.5
)

ci_upper = np.percentile(
    bootstrap_beta1,
    97.5
)

bias_cc = beta1_cc - beta1_true

print("MCAR Complete Case")
print("beta1 =", beta1_cc)
print("SE =", se_cc)
print("95% CI =", (ci_lower, ci_upper))
print("Bias =", bias_cc)
print("Successful bootstrap samples =", len(bootstrap_beta1))
save_master_result("Complete Case", "MCAR", beta1_cc, se_cc, ci_lower, ci_upper, bias_cc)


MCAR Complete Case
beta1 = 1.7847226108996361
SE = 0.13553333590610614
95% CI = (np.float64(1.5272091458782524), np.float64(2.0247651454461844))
Bias = -0.21527738910036387
Successful bootstrap samples = 500


In [ ]:
# ==========================================
# STEP 6B: MAR + COMPLETE CASE
# ==========================================

df_mar_cc = df_mar.dropna()


X_mar_cc = sm.add_constant(
    df_mar_cc[["X1", "X2", "X3"]]
)


results_mar_cc = sm.OLS(
    df_mar_cc["Y"],
    X_mar_cc
).fit()


print("\n===== MAR + COMPLETE CASE =====")

print(
    "Observations:",
    len(df_mar_cc)
)

print(
    "Estimated coefficients:"
)

print(results_mar_cc.params)
save_point_estimates("Complete Case", "MAR", results_mar_cc.params)


In [538]:
# ============================================
# MAR: COMPLETE CASE + CORRECTED BOOTSTRAP
# ============================================

df = df_mar.copy()

# Original complete-case estimate
complete_cases = df.dropna(
    subset=["X1", "X2", "X3", "Y"]
).reset_index(drop=True)

X_original = sm.add_constant(
    complete_cases[["X1", "X2", "X3"]]
)

model_original = sm.OLS(
    complete_cases["Y"],
    X_original
).fit()

beta1_cc = model_original.params["X1"]

# --------------------------------------------
# Bootstrap
# --------------------------------------------

B = 500
bootstrap_beta1 = []

np.random.seed(123)

for b in range(B):

    # Resample from ALL 100 original rows
    sample_indices = np.random.choice(
        len(df),
        size=len(df),
        replace=True
    )

    bootstrap_sample = df.iloc[
        sample_indices
    ].reset_index(drop=True)

    # Apply complete-case analysis
    # inside the bootstrap sample
    bootstrap_complete = bootstrap_sample.dropna(
        subset=["X1", "X2", "X3", "Y"]
    )

    if len(bootstrap_complete) <= 4:
        continue

    X_boot = sm.add_constant(
        bootstrap_complete[["X1", "X2", "X3"]]
    )

    model_boot = sm.OLS(
        bootstrap_complete["Y"],
        X_boot
    ).fit()

    bootstrap_beta1.append(
        model_boot.params["X1"]
    )

bootstrap_beta1 = np.array(bootstrap_beta1)

# Bootstrap inference
se_cc = np.std(
    bootstrap_beta1,
    ddof=1
)

ci_lower = np.percentile(
    bootstrap_beta1,
    2.5
)

ci_upper = np.percentile(
    bootstrap_beta1,
    97.5
)

bias_cc = beta1_cc - beta1_true

print("MAR Complete Case")
print("beta1 =", beta1_cc)
print("SE =", se_cc)
print("95% CI =", (ci_lower, ci_upper))
print("Bias =", bias_cc)
print("Successful bootstrap samples =", len(bootstrap_beta1))
save_master_result("Complete Case", "MAR", beta1_cc, se_cc, ci_lower, ci_upper, bias_cc)


MAR Complete Case
beta1 = 1.7768964817621882
SE = 0.13603742501551958
95% CI = (np.float64(1.532446588100685), np.float64(2.0554792572194347))
Bias = -0.22310351823781183
Successful bootstrap samples = 500


In [ ]:
# ==========================================
# STEP 6C: MNAR + COMPLETE CASE
# ==========================================

df_mnar_cc = df_mnar.dropna()


X_mnar_cc = sm.add_constant(
    df_mnar_cc[["X1", "X2", "X3"]]
)


results_mnar_cc = sm.OLS(
    df_mnar_cc["Y"],
    X_mnar_cc
).fit()


print("\n===== MNAR + COMPLETE CASE =====")

print(
    "Observations:",
    len(df_mnar_cc)
)

print(
    "Estimated coefficients:"
)

print(results_mnar_cc.params)
save_point_estimates("Complete Case", "MNAR", results_mnar_cc.params)


In [540]:
# ============================================
# MNAR: COMPLETE CASE + CORRECTED BOOTSTRAP
# ============================================

df = df_mnar.copy()

# Original complete-case estimate
complete_cases = df.dropna(
    subset=["X1", "X2", "X3", "Y"]
).reset_index(drop=True)

X_original = sm.add_constant(
    complete_cases[["X1", "X2", "X3"]]
)

model_original = sm.OLS(
    complete_cases["Y"],
    X_original
).fit()

beta1_cc = model_original.params["X1"]

# --------------------------------------------
# Bootstrap
# --------------------------------------------

B = 500
bootstrap_beta1 = []

np.random.seed(123)

for b in range(B):

    # Resample from ALL 100 original rows
    sample_indices = np.random.choice(
        len(df),
        size=len(df),
        replace=True
    )

    bootstrap_sample = df.iloc[
        sample_indices
    ].reset_index(drop=True)

    # Apply complete-case analysis
    # inside the bootstrap sample
    bootstrap_complete = bootstrap_sample.dropna(
        subset=["X1", "X2", "X3", "Y"]
    )

    if len(bootstrap_complete) <= 4:
        continue

    X_boot = sm.add_constant(
        bootstrap_complete[["X1", "X2", "X3"]]
    )

    model_boot = sm.OLS(
        bootstrap_complete["Y"],
        X_boot
    ).fit()

    bootstrap_beta1.append(
        model_boot.params["X1"]
    )

bootstrap_beta1 = np.array(bootstrap_beta1)

# Bootstrap inference
se_cc = np.std(
    bootstrap_beta1,
    ddof=1
)

ci_lower = np.percentile(
    bootstrap_beta1,
    2.5
)

ci_upper = np.percentile(
    bootstrap_beta1,
    97.5
)

bias_cc = beta1_cc - beta1_true

print("MNAR Complete Case")
print("beta1 =", beta1_cc)
print("SE =", se_cc)
print("95% CI =", (ci_lower, ci_upper))
print("Bias =", bias_cc)
print("Successful bootstrap samples =", len(bootstrap_beta1))
save_master_result("Complete Case", "MNAR", beta1_cc, se_cc, ci_lower, ci_upper, bias_cc)


MNAR Complete Case
beta1 = 1.7486858337955549
SE = 0.1219569274735909
95% CI = (np.float64(1.5008796538689382), np.float64(1.981103279238143))
Bias = -0.25131416620444513
Successful bootstrap samples = 500


In [ ]:
# ==========================================
# STEP 7A: MCAR + MEAN IMPUTATION
# ==========================================

# Calculate observed Y mean
mean_y_mcar = df_mcar["Y"].mean()


# Copy MCAR data
df_mcar_mean = df_mcar.copy()


# Fill missing Y with observed mean
df_mcar_mean["Y"] = (
    df_mcar_mean["Y"].fillna(mean_y_mcar)
)


# Regression using X1, X2, X3
X_mcar_mean = sm.add_constant(
    df_mcar_mean[["X1", "X2", "X3"]]
)


results_mcar_mean = sm.OLS(
    df_mcar_mean["Y"],
    X_mcar_mean
).fit()


# X1 coefficient
beta1_hat_mcar_mean = (
    results_mcar_mean.params["X1"]
)

se_beta1_mcar_mean = (
    results_mcar_mean.bse["X1"]
)

ci_beta1_mcar_mean = (
    results_mcar_mean.conf_int().loc["X1"]
)

bias_mcar_mean = (
    beta1_hat_mcar_mean - beta1_true
)


print("\n===== MCAR + MEAN =====")

print(
    "Estimated beta1:",
    beta1_hat_mcar_mean
)

print(
    "Standard error:",
    se_beta1_mcar_mean
)

print(
    "95% CI:",
    tuple(ci_beta1_mcar_mean)
)

print(
    "Bias:",
    bias_mcar_mean
)
save_point_estimates("Mean Imputation", "MCAR", results_mcar_mean.params)


In [542]:
# ============================================
# MCAR: MEAN IMPUTATION + BOOTSTRAP
# ============================================

df = df_mcar.copy()

B = 500
bootstrap_beta1 = []

np.random.seed(123)

for b in range(B):

    # Resample rows with replacement
    sample_indices = np.random.choice(
        len(df),
        size=len(df),
        replace=True
    )

    bootstrap_sample = df.iloc[sample_indices].reset_index(drop=True)

    # Mean imputation
    mean_y = bootstrap_sample["Y"].mean()

    bootstrap_sample["Y"] = bootstrap_sample["Y"].fillna(mean_y)

    # Regression after imputation
    X_boot = sm.add_constant(
        bootstrap_sample[["X1", "X2", "X3"]]
    )

    model_boot = sm.OLS(
        bootstrap_sample["Y"],
        X_boot
    ).fit()

    # Store beta1
    bootstrap_beta1.append(
        model_boot.params["X1"]
    )

bootstrap_beta1 = np.array(bootstrap_beta1)

# Original mean-imputation estimate
mean_y_original = df["Y"].mean()

df_completed = df.copy()
df_completed["Y"] = df_completed["Y"].fillna(mean_y_original)

X_original = sm.add_constant(
    df_completed[["X1", "X2", "X3"]]
)

model_original = sm.OLS(
    df_completed["Y"],
    X_original
).fit()

beta1_mean = model_original.params["X1"]

# Bootstrap inference
se_mean = np.std(
    bootstrap_beta1,
    ddof=1
)

ci_lower = np.percentile(
    bootstrap_beta1,
    2.5
)

ci_upper = np.percentile(
    bootstrap_beta1,
    97.5
)

bias_mean = beta1_mean - 2

print("MCAR Mean Imputation")
print("beta1 =", beta1_mean)
print("SE =", se_mean)
print("95% CI =", (ci_lower, ci_upper))
print("Bias =", bias_mean)
save_master_result("Mean Imputation", "MCAR", beta1_mean, se_mean, ci_lower, ci_upper, bias_mean)


MCAR Mean Imputation
beta1 = 1.08874973196683
SE = 0.2285332645448801
95% CI = (np.float64(0.6490417498051037), np.float64(1.564978352361869))
Bias = -0.91125026803317


In [ ]:
# ==========================================
# STEP 7B: MAR + MEAN IMPUTATION
# ==========================================

mean_y_mar = df_mar["Y"].mean()


df_mar_mean = df_mar.copy()


df_mar_mean["Y"] = (
    df_mar_mean["Y"].fillna(mean_y_mar)
)


X_mar_mean = sm.add_constant(
    df_mar_mean[["X1", "X2", "X3"]]
)


results_mar_mean = sm.OLS(
    df_mar_mean["Y"],
    X_mar_mean
).fit()


beta1_hat_mar_mean = (
    results_mar_mean.params["X1"]
)

se_beta1_mar_mean = (
    results_mar_mean.bse["X1"]
)

ci_beta1_mar_mean = (
    results_mar_mean.conf_int().loc["X1"]
)

bias_mar_mean = (
    beta1_hat_mar_mean - beta1_true
)


print("\n===== MAR + MEAN =====")

print(
    "Estimated beta1:",
    beta1_hat_mar_mean
)

print(
    "Standard error:",
    se_beta1_mar_mean
)

print(
    "95% CI:",
    tuple(ci_beta1_mar_mean)
)

print(
    "Bias:",
    bias_mar_mean
)


save_point_estimates("Mean Imputation", "MAR", results_mar_mean.params)


In [544]:
# ============================================
# MAR: MEAN IMPUTATION + BOOTSTRAP
# ============================================

df = df_mar.copy()

B = 500
bootstrap_beta1 = []

np.random.seed(123)

for b in range(B):

    # Resample rows with replacement
    sample_indices = np.random.choice(
        len(df),
        size=len(df),
        replace=True
    )

    bootstrap_sample = df.iloc[
        sample_indices
    ].reset_index(drop=True)

    # Mean imputation
    mean_y = bootstrap_sample["Y"].mean()

    bootstrap_sample["Y"] = bootstrap_sample[
        "Y"
    ].fillna(mean_y)

    # Regression after imputation
    X_boot = sm.add_constant(
        bootstrap_sample[["X1", "X2", "X3"]]
    )

    model_boot = sm.OLS(
        bootstrap_sample["Y"],
        X_boot
    ).fit()

    # Store beta1
    bootstrap_beta1.append(
        model_boot.params["X1"]
    )

bootstrap_beta1 = np.array(bootstrap_beta1)

# Original mean-imputation estimate
mean_y_original = df["Y"].mean()

df_completed = df.copy()
df_completed["Y"] = df_completed["Y"].fillna(
    mean_y_original
)

X_original = sm.add_constant(
    df_completed[["X1", "X2", "X3"]]
)

model_original = sm.OLS(
    df_completed["Y"],
    X_original
).fit()

beta1_mean = model_original.params["X1"]

# Bootstrap inference
se_mean = np.std(
    bootstrap_beta1,
    ddof=1
)

ci_lower = np.percentile(
    bootstrap_beta1,
    2.5
)

ci_upper = np.percentile(
    bootstrap_beta1,
    97.5
)

bias_mean = beta1_mean - 2

print("MAR Mean Imputation")
print("beta1 =", beta1_mean)
print("SE =", se_mean)
print("95% CI =", (ci_lower, ci_upper))
print("Bias =", bias_mean)
save_master_result("Mean Imputation", "MAR", beta1_mean, se_mean, ci_lower, ci_upper, bias_mean)


MAR Mean Imputation
beta1 = 1.3065738803814224
SE = 0.18635987757541259
95% CI = (np.float64(0.9152749342769897), np.float64(1.6593808604317717))
Bias = -0.6934261196185776


In [ ]:
# ==========================================
# STEP 7C: MNAR + MEAN IMPUTATION
# ==========================================

mean_y_mnar = df_mnar["Y"].mean()


df_mnar_mean = df_mnar.copy()


df_mnar_mean["Y"] = (
    df_mnar_mean["Y"].fillna(mean_y_mnar)
)


X_mnar_mean = sm.add_constant(
    df_mnar_mean[["X1", "X2", "X3"]]
)


results_mnar_mean = sm.OLS(
    df_mnar_mean["Y"],
    X_mnar_mean
).fit()


beta1_hat_mnar_mean = (
    results_mnar_mean.params["X1"]
)

se_beta1_mnar_mean = (
    results_mnar_mean.bse["X1"]
)

ci_beta1_mnar_mean = (
    results_mnar_mean.conf_int().loc["X1"]
)

bias_mnar_mean = (
    beta1_hat_mnar_mean - beta1_true
)


print("\n===== MNAR + MEAN =====")

print(
    "Estimated beta1:",
    beta1_hat_mnar_mean
)

print(
    "Standard error:",
    se_beta1_mnar_mean
)

print(
    "95% CI:",
    tuple(ci_beta1_mnar_mean)
)

print(
    "Bias:",
    bias_mnar_mean
)

save_point_estimates("Mean Imputation", "MNAR", results_mnar_mean.params)


In [546]:
# ============================================
# MNAR: MEAN IMPUTATION + BOOTSTRAP
# ============================================

df = df_mnar.copy()

B = 500
bootstrap_beta1 = []

np.random.seed(123)

for b in range(B):

    # Resample rows with replacement
    sample_indices = np.random.choice(
        len(df),
        size=len(df),
        replace=True
    )

    bootstrap_sample = df.iloc[
        sample_indices
    ].reset_index(drop=True)

    # Mean imputation
    mean_y = bootstrap_sample["Y"].mean()

    bootstrap_sample["Y"] = bootstrap_sample[
        "Y"
    ].fillna(mean_y)

    # Regression after imputation
    X_boot = sm.add_constant(
        bootstrap_sample[["X1", "X2", "X3"]]
    )

    model_boot = sm.OLS(
        bootstrap_sample["Y"],
        X_boot
    ).fit()

    # Store beta1
    bootstrap_beta1.append(
        model_boot.params["X1"]
    )

bootstrap_beta1 = np.array(bootstrap_beta1)

# Original mean-imputation estimate
mean_y_original = df["Y"].mean()

df_completed = df.copy()
df_completed["Y"] = df_completed["Y"].fillna(
    mean_y_original
)

X_original = sm.add_constant(
    df_completed[["X1", "X2", "X3"]]
)

model_original = sm.OLS(
    df_completed["Y"],
    X_original
).fit()

beta1_mean = model_original.params["X1"]

# Bootstrap inference
se_mean = np.std(
    bootstrap_beta1,
    ddof=1
)

ci_lower = np.percentile(
    bootstrap_beta1,
    2.5
)

ci_upper = np.percentile(
    bootstrap_beta1,
    97.5
)

bias_mean = beta1_mean - 2

print("MNAR Mean Imputation")
print("beta1 =", beta1_mean)
print("SE =", se_mean)
print("95% CI =", (ci_lower, ci_upper))
print("Bias =", bias_mean)
save_master_result("Mean Imputation", "MNAR", beta1_mean, se_mean, ci_lower, ci_upper, bias_mean)


MNAR Mean Imputation
beta1 = 1.412745233675645
SE = 0.16992419172385081
95% CI = (np.float64(1.0471923265387149), np.float64(1.7019818494141112))
Bias = -0.587254766324355


In [ ]:
# ==========================================
# STEP 8A: MCAR + HOT-DECK
# ==========================================

# Make a copy
df_mcar_hotdeck = df_mcar.copy()


# Separate observed and missing rows
observed_mcar = df_mcar[
    df_mcar["Y"].notna()
]

missing_mcar = df_mcar[
    df_mcar["Y"].isna()
]


print("\n===== MCAR HOT-DECK =====")

print(
    "Observed Y values:",
    len(observed_mcar)
)

print(
    "Missing Y values:",
    len(missing_mcar)
)


# Predictors used to find donors
predictors = ["X1", "X2", "X3"]


# Find a donor for every missing observation
for idx in missing_mcar.index:

    # Predictor values of missing observation
    x_missing = df_mcar.loc[
        idx,
        predictors
    ]

    # Squared distance from every observed row
    distances = (
        (
            observed_mcar[predictors]
            - x_missing
        ) ** 2
    ).sum(axis=1)

    # Index of closest donor
    donor_idx = distances.idxmin()

    # Donor's observed Y
    donor_y = observed_mcar.loc[
        donor_idx,
        "Y"
    ]

    # Fill missing Y
    df_mcar_hotdeck.loc[
        idx,
        "Y"
    ] = donor_y


print(
    "Missing Y after hot-deck:",
    df_mcar_hotdeck["Y"].isna().sum()
)


# Regression
X_mcar_hotdeck = sm.add_constant(
    df_mcar_hotdeck[predictors]
)


results_mcar_hotdeck = sm.OLS(
    df_mcar_hotdeck["Y"],
    X_mcar_hotdeck
).fit()


# Results for beta1
beta1_hat_mcar_hotdeck = (
    results_mcar_hotdeck.params["X1"]
)

se_beta1_mcar_hotdeck = (
    results_mcar_hotdeck.bse["X1"]
)

ci_beta1_mcar_hotdeck = (
    results_mcar_hotdeck.conf_int().loc["X1"]
)

bias_mcar_hotdeck = (
    beta1_hat_mcar_hotdeck - beta1_true
)


print("\n===== MCAR + HOT-DECK =====")

print(
    "Estimated beta1:",
    beta1_hat_mcar_hotdeck
)

print(
    "Standard error:",
    se_beta1_mcar_hotdeck
)

print(
    "95% CI:",
    tuple(ci_beta1_mcar_hotdeck)
)

print(
    "Bias:",
    bias_mcar_hotdeck
)
save_point_estimates("Hot-Deck", "MCAR", results_mcar_hotdeck.params)


In [548]:
# ============================================
# MCAR: HOT-DECK + BOOTSTRAP
# ============================================

df = df_mcar.copy()

predictors = ["X1", "X2", "X3"]

B = 500
bootstrap_beta1 = []

np.random.seed(123)

for b in range(B):

    # Resample rows with replacement
    sample_indices = np.random.choice(
        len(df),
        size=len(df),
        replace=True
    )

    bootstrap_sample = df.iloc[
        sample_indices
    ].reset_index(drop=True)

    # Identify observed and missing Y
    observed = bootstrap_sample["Y"].notna()
    missing = bootstrap_sample["Y"].isna()

    # Make a copy for hot-deck imputation
    completed = bootstrap_sample.copy()

    # Observed donor rows
    observed_data = bootstrap_sample.loc[
        observed
    ]

    # Hot-deck: find nearest donor for each missing Y
    for idx in bootstrap_sample.index[missing]:

        x_missing = bootstrap_sample.loc[
            idx, predictors
        ]

        distances = (
            (
                observed_data[predictors]
                - x_missing
            ) ** 2
        ).sum(axis=1)

        donor_idx = distances.idxmin()

        completed.loc[idx, "Y"] = (
            observed_data.loc[donor_idx, "Y"]
        )

    # Regression after hot-deck imputation
    X_boot = sm.add_constant(
        completed[["X1", "X2", "X3"]]
    )

    model_boot = sm.OLS(
        completed["Y"],
        X_boot
    ).fit()

    bootstrap_beta1.append(
        model_boot.params["X1"]
    )

bootstrap_beta1 = np.array(
    bootstrap_beta1
)

# --------------------------------------------
# Original hot-deck estimate
# --------------------------------------------

observed = df["Y"].notna()
missing = df["Y"].isna()

completed_original = df.copy()

observed_data = df.loc[observed]

for idx in df.index[missing]:

    x_missing = df.loc[idx, predictors]

    distances = (
        (
            observed_data[predictors]
            - x_missing
        ) ** 2
    ).sum(axis=1)

    donor_idx = distances.idxmin()

    completed_original.loc[idx, "Y"] = (
        observed_data.loc[donor_idx, "Y"]
    )

X_original = sm.add_constant(
    completed_original[["X1", "X2", "X3"]]
)

model_original = sm.OLS(
    completed_original["Y"],
    X_original
).fit()

beta1_hotdeck = model_original.params["X1"]

# Bootstrap inference
se_hotdeck = np.std(
    bootstrap_beta1,
    ddof=1
)

ci_lower = np.percentile(
    bootstrap_beta1,
    2.5
)

ci_upper = np.percentile(
    bootstrap_beta1,
    97.5
)

bias_hotdeck = beta1_hotdeck - 2

print("MCAR Hot-deck")
print("beta1 =", beta1_hotdeck)
print("SE =", se_hotdeck)
print("95% CI =", (ci_lower, ci_upper))
print("Bias =", bias_hotdeck)
save_master_result("Hot-Deck", "MCAR", beta1_hotdeck, se_hotdeck, ci_lower, ci_upper, bias_hotdeck)


MCAR Hot-deck
beta1 = 1.6583060087226391
SE = 0.16541662071804714
95% CI = (np.float64(1.2618362825881524), np.float64(1.9184228178030354))
Bias = -0.3416939912773609


In [ ]:
# ==========================================
# STEP 8B: MAR + HOT-DECK
# ==========================================

df_mar_hotdeck = df_mar.copy()


observed_mar = df_mar[
    df_mar["Y"].notna()
]

missing_mar = df_mar[
    df_mar["Y"].isna()
]


print("\n===== MAR HOT-DECK =====")

print(
    "Observed Y values:",
    len(observed_mar)
)

print(
    "Missing Y values:",
    len(missing_mar)
)


predictors = ["X1", "X2", "X3"]


for idx in missing_mar.index:

    # X1, X2, X3 of missing observation
    x_missing = df_mar.loc[
        idx,
        predictors
    ]

    # Distance from every observed row
    distances = (
        (
            observed_mar[predictors]
            - x_missing
        ) ** 2
    ).sum(axis=1)

    # Closest donor
    donor_idx = distances.idxmin()

    # Donor's Y
    donor_y = observed_mar.loc[
        donor_idx,
        "Y"
    ]

    # Fill missing Y
    df_mar_hotdeck.loc[
        idx,
        "Y"
    ] = donor_y


print(
    "Missing Y after hot-deck:",
    df_mar_hotdeck["Y"].isna().sum()
)


# Regression
X_mar_hotdeck = sm.add_constant(
    df_mar_hotdeck[predictors]
)


results_mar_hotdeck = sm.OLS(
    df_mar_hotdeck["Y"],
    X_mar_hotdeck
).fit()


# Results
beta1_hat_mar_hotdeck = (
    results_mar_hotdeck.params["X1"]
)

se_beta1_mar_hotdeck = (
    results_mar_hotdeck.bse["X1"]
)

ci_beta1_mar_hotdeck = (
    results_mar_hotdeck.conf_int().loc["X1"]
)

bias_mar_hotdeck = (
    beta1_hat_mar_hotdeck - beta1_true
)


print("\n===== MAR + HOT-DECK =====")

print(
    "Estimated beta1:",
    beta1_hat_mar_hotdeck
)

print(
    "Standard error:",
    se_beta1_mar_hotdeck
)

print(
    "95% CI:",
    tuple(ci_beta1_mar_hotdeck)
)

print(
    "Bias:",
    bias_mar_hotdeck
)


save_point_estimates("Hot-Deck", "MAR", results_mar_hotdeck.params)


In [550]:
# ============================================
# MAR: HOT-DECK + BOOTSTRAP
# ============================================

df = df_mar.copy()

predictors = ["X1", "X2", "X3"]

B = 500
bootstrap_beta1 = []

np.random.seed(123)

for b in range(B):

    # Resample rows with replacement
    sample_indices = np.random.choice(
        len(df),
        size=len(df),
        replace=True
    )

    bootstrap_sample = df.iloc[
        sample_indices
    ].reset_index(drop=True)

    # Identify observed and missing Y
    observed = bootstrap_sample["Y"].notna()
    missing = bootstrap_sample["Y"].isna()

    completed = bootstrap_sample.copy()

    # Observed donor rows
    observed_data = bootstrap_sample.loc[
        observed
    ]

    # Hot-deck imputation
    for idx in bootstrap_sample.index[missing]:

        x_missing = bootstrap_sample.loc[
            idx, predictors
        ]

        distances = (
            (
                observed_data[predictors]
                - x_missing
            ) ** 2
        ).sum(axis=1)

        donor_idx = distances.idxmin()

        completed.loc[idx, "Y"] = (
            observed_data.loc[donor_idx, "Y"]
        )

    # Regression after imputation
    X_boot = sm.add_constant(
        completed[["X1", "X2", "X3"]]
    )

    model_boot = sm.OLS(
        completed["Y"],
        X_boot
    ).fit()

    bootstrap_beta1.append(
        model_boot.params["X1"]
    )

bootstrap_beta1 = np.array(
    bootstrap_beta1
)

# --------------------------------------------
# Original hot-deck estimate
# --------------------------------------------

observed = df["Y"].notna()
missing = df["Y"].isna()

completed_original = df.copy()

observed_data = df.loc[observed]

for idx in df.index[missing]:

    x_missing = df.loc[idx, predictors]

    distances = (
        (
            observed_data[predictors]
            - x_missing
        ) ** 2
    ).sum(axis=1)

    donor_idx = distances.idxmin()

    completed_original.loc[idx, "Y"] = (
        observed_data.loc[donor_idx, "Y"]
    )

X_original = sm.add_constant(
    completed_original[["X1", "X2", "X3"]]
)

model_original = sm.OLS(
    completed_original["Y"],
    X_original
).fit()

beta1_hotdeck = model_original.params["X1"]

# Bootstrap inference
se_hotdeck = np.std(
    bootstrap_beta1,
    ddof=1
)

ci_lower = np.percentile(
    bootstrap_beta1,
    2.5
)

ci_upper = np.percentile(
    bootstrap_beta1,
    97.5
)

bias_hotdeck = beta1_hotdeck - 2

print("MAR Hot-deck")
print("beta1 =", beta1_hotdeck)
print("SE =", se_hotdeck)
print("95% CI =", (ci_lower, ci_upper))
print("Bias =", bias_hotdeck)
save_master_result("Hot-Deck", "MAR", beta1_hotdeck, se_hotdeck, ci_lower, ci_upper, bias_hotdeck)


MAR Hot-deck
beta1 = 1.7871541193093785
SE = 0.14802852327219387
95% CI = (np.float64(1.41639645335674), np.float64(1.9961012035770664))
Bias = -0.2128458806906215


In [ ]:
# ==========================================
# STEP 8C: MNAR + HOT-DECK
# ==========================================

df_mnar_hotdeck = df_mnar.copy()


observed_mnar = df_mnar[
    df_mnar["Y"].notna()
]

missing_mnar = df_mnar[
    df_mnar["Y"].isna()
]


print("\n===== MNAR HOT-DECK =====")

print(
    "Observed Y values:",
    len(observed_mnar)
)

print(
    "Missing Y values:",
    len(missing_mnar)
)


predictors = ["X1", "X2", "X3"]


for idx in missing_mnar.index:

    # X1, X2, X3 of missing observation
    x_missing = df_mnar.loc[
        idx,
        predictors
    ]

    # Distance from every observed row
    distances = (
        (
            observed_mnar[predictors]
            - x_missing
        ) ** 2
    ).sum(axis=1)

    # Closest donor
    donor_idx = distances.idxmin()

    # Donor's Y
    donor_y = observed_mnar.loc[
        donor_idx,
        "Y"
    ]

    # Fill missing Y
    df_mnar_hotdeck.loc[
        idx,
        "Y"
    ] = donor_y


print(
    "Missing Y after hot-deck:",
    df_mnar_hotdeck["Y"].isna().sum()
)


# Regression
X_mnar_hotdeck = sm.add_constant(
    df_mnar_hotdeck[predictors]
)


results_mnar_hotdeck = sm.OLS(
    df_mnar_hotdeck["Y"],
    X_mnar_hotdeck
).fit()


# Results
beta1_hat_mnar_hotdeck = (
    results_mnar_hotdeck.params["X1"]
)

se_beta1_mnar_hotdeck = (
    results_mnar_hotdeck.bse["X1"]
)

ci_beta1_mnar_hotdeck = (
    results_mnar_hotdeck.conf_int().loc["X1"]
)

bias_mnar_hotdeck = (
    beta1_hat_mnar_hotdeck - beta1_true
)


print("\n===== MNAR + HOT-DECK =====")

print(
    "Estimated beta1:",
    beta1_hat_mnar_hotdeck
)

print(
    "Standard error:",
    se_beta1_mnar_hotdeck
)

print(
    "95% CI:",
    tuple(ci_beta1_mnar_hotdeck)
)

print(
    "Bias:",
    bias_mnar_hotdeck
)
save_point_estimates("Hot-Deck", "MNAR", results_mnar_hotdeck.params)


In [552]:
# ============================================
# MNAR: HOT-DECK + BOOTSTRAP
# ============================================

df = df_mnar.copy()

predictors = ["X1", "X2", "X3"]

B = 500
bootstrap_beta1 = []

np.random.seed(123)

for b in range(B):

    # Resample rows with replacement
    sample_indices = np.random.choice(
        len(df),
        size=len(df),
        replace=True
    )

    bootstrap_sample = df.iloc[
        sample_indices
    ].reset_index(drop=True)

    # Identify observed and missing Y
    observed = bootstrap_sample["Y"].notna()
    missing = bootstrap_sample["Y"].isna()

    completed = bootstrap_sample.copy()

    # Observed donor rows
    observed_data = bootstrap_sample.loc[
        observed
    ]

    # Hot-deck imputation
    for idx in bootstrap_sample.index[missing]:

        x_missing = bootstrap_sample.loc[
            idx, predictors
        ]

        distances = (
            (
                observed_data[predictors]
                - x_missing
            ) ** 2
        ).sum(axis=1)

        donor_idx = distances.idxmin()

        completed.loc[idx, "Y"] = (
            observed_data.loc[donor_idx, "Y"]
        )

    # Regression after imputation
    X_boot = sm.add_constant(
        completed[["X1", "X2", "X3"]]
    )

    model_boot = sm.OLS(
        completed["Y"],
        X_boot
    ).fit()

    bootstrap_beta1.append(
        model_boot.params["X1"]
    )

bootstrap_beta1 = np.array(
    bootstrap_beta1
)

# --------------------------------------------
# Original hot-deck estimate
# --------------------------------------------

observed = df["Y"].notna()
missing = df["Y"].isna()

completed_original = df.copy()

observed_data = df.loc[observed]

for idx in df.index[missing]:

    x_missing = df.loc[idx, predictors]

    distances = (
        (
            observed_data[predictors]
            - x_missing
        ) ** 2
    ).sum(axis=1)

    donor_idx = distances.idxmin()

    completed_original.loc[idx, "Y"] = (
        observed_data.loc[donor_idx, "Y"]
    )

X_original = sm.add_constant(
    completed_original[["X1", "X2", "X3"]]
)

model_original = sm.OLS(
    completed_original["Y"],
    X_original
).fit()

beta1_hotdeck = model_original.params["X1"]

# Bootstrap inference
se_hotdeck = np.std(
    bootstrap_beta1,
    ddof=1
)

ci_lower = np.percentile(
    bootstrap_beta1,
    2.5
)

ci_upper = np.percentile(
    bootstrap_beta1,
    97.5
)

bias_hotdeck = beta1_hotdeck - 2

print("MNAR Hot-deck")
print("beta1 =", beta1_hotdeck)
print("SE =", se_hotdeck)
print("95% CI =", (ci_lower, ci_upper))
print("Bias =", bias_hotdeck)
save_master_result("Hot-Deck", "MNAR", beta1_hotdeck, se_hotdeck, ci_lower, ci_upper, bias_hotdeck)


MNAR Hot-deck
beta1 = 1.7222065918217586
SE = 0.12336767983306333
95% CI = (np.float64(1.468632992252233), np.float64(1.9535506990662808))
Bias = -0.2777934081782414


In [553]:
# ==========================================
# STEP 9A: MCAR + STOCHASTIC REGRESSION
#           FIT MODEL ON OBSERVED Y
# ==========================================

# Make a copy of the MCAR data
df_mcar_stoch = df_mcar.copy()

# Keep only rows where Y is observed
observed_mcar = df_mcar[
    df_mcar["Y"].notna()
]

# Predictors
predictors = ["X1", "X2", "X3"]

# Add intercept
X_mcar_obs = sm.add_constant(
    observed_mcar[predictors]
)

# Fit regression using observed Y
model_mcar_stoch = sm.OLS(
    observed_mcar["Y"],
    X_mcar_obs
)

results_mcar_stoch = (
    model_mcar_stoch.fit()
)

# Get fitted residuals
residuals_mcar = (
    results_mcar_stoch.resid
)

print("Observed Y values:", len(observed_mcar))
print("Number of residuals:", len(residuals_mcar))

Observed Y values: 75
Number of residuals: 75


In [554]:
# ==========================================
# STEP 9B: PREDICT THE MISSING Y VALUES
# ==========================================

# Rows where Y is missing
missing_mcar = df_mcar[
    df_mcar["Y"].isna()
]

# Add intercept to their predictors
X_mcar_missing = sm.add_constant(
    missing_mcar[predictors]
)

# Predict expected Y for each missing row
predicted_y_mcar = (
    results_mcar_stoch.predict(
        X_mcar_missing
    )
)

print("Number of missing Y values:", len(missing_mcar))

print("\nPredicted Y values:")
print(predicted_y_mcar.head())

Number of missing Y values: 25

Predicted Y values:
2     7.755027
13   -1.755708
20    8.652005
30    5.614003
31    8.361945
dtype: float64


In [555]:
# ==========================================
# STEP 9C: ADD RANDOM RESIDUALS
# ==========================================

# Set seed so this stochastic imputation
# can be reproduced
np.random.seed(42)

# Randomly select residuals from the observed
# residuals, with replacement
random_residuals_mcar = np.random.choice(
    residuals_mcar,
    size=len(missing_mcar),
    replace=True
)

# Create stochastic imputations
stochastic_y_mcar = (
    predicted_y_mcar.to_numpy()
    + random_residuals_mcar
)

print("\nPredicted Y:")
print(predicted_y_mcar.head())

print("\nRandom residuals:")
print(random_residuals_mcar[:5])

print("\nStochastic imputed Y:")
print(stochastic_y_mcar[:5])


Predicted Y:
2     7.755027
13   -1.755708
20    8.652005
30    5.614003
31    8.361945
dtype: float64

Random residuals:
[-0.50952043  0.52979245 -1.70395456 -0.56972111  1.72591728]

Stochastic imputed Y:
[ 7.24550645 -1.22591598  6.9480509   5.04428147 10.08786246]


In [556]:
# ==========================================
# STEP 9D: FILL THE MISSING Y VALUES
# ==========================================

# Get the indices of missing observations
missing_indices_mcar = missing_mcar.index

# Fill the missing Y values
df_mcar_stoch.loc[
    missing_indices_mcar,
    "Y"
] = stochastic_y_mcar


# Check
print(
    "Missing Y after stochastic regression:",
    df_mcar_stoch["Y"].isna().sum()
)

Missing Y after stochastic regression: 0


In [ ]:
# ==========================================
# STEP 9E: REGRESSION AFTER
#          STOCHASTIC IMPUTATION
# ==========================================

X_mcar_stoch_final = sm.add_constant(
    df_mcar_stoch[predictors]
)

results_mcar_stoch_final = sm.OLS(
    df_mcar_stoch["Y"],
    X_mcar_stoch_final
).fit()


# X1 coefficient
beta1_hat_mcar_stoch = (
    results_mcar_stoch_final.params["X1"]
)

# Standard error
se_beta1_mcar_stoch = (
    results_mcar_stoch_final.bse["X1"]
)

# 95% confidence interval
ci_beta1_mcar_stoch = (
    results_mcar_stoch_final.conf_int()
    .loc["X1"]
)

# Bias
bias_mcar_stoch = (
    beta1_hat_mcar_stoch - beta1_true
)


print("\n===== MCAR + STOCHASTIC REGRESSION =====")

print(
    "True beta1:",
    beta1_true
)

print(
    "Estimated beta1:",
    beta1_hat_mcar_stoch
)

print(
    "Standard error:",
    se_beta1_mcar_stoch
)

print(
    "95% CI:",
    tuple(ci_beta1_mcar_stoch)
)

print(
    "Bias:",
    bias_mcar_stoch
)
save_point_estimates("Stochastic Regression", "MCAR", results_mcar_stoch_final.params)


In [558]:
# ============================================
# MCAR: STOCHASTIC REGRESSION + BOOTSTRAP
# ============================================

df = df_mcar.copy()

B = 500
bootstrap_beta1 = []

np.random.seed(123)

for b in range(B):

    # Resample rows with replacement
    sample_indices = np.random.choice(
        len(df),
        size=len(df),
        replace=True
    )

    bootstrap_sample = df.iloc[
        sample_indices
    ].reset_index(drop=True)

    observed = bootstrap_sample["Y"].notna()
    missing = bootstrap_sample["Y"].isna()

    # Design matrix
    X_boot = sm.add_constant(
        bootstrap_sample[["X1", "X2", "X3"]]
    )

    # Fit regression using observed Y
    model_obs = sm.OLS(
        bootstrap_sample.loc[observed, "Y"],
        X_boot.loc[observed]
    ).fit()

    # Predict Y
    y_hat = model_obs.predict(X_boot)

    # Observed residuals
    residuals = model_obs.resid.to_numpy()

    # Start with bootstrap sample
    completed = bootstrap_sample.copy()

    # Stochastic imputation
    for idx in bootstrap_sample.index[missing]:

        random_residual = np.random.choice(
            residuals
        )

        completed.loc[idx, "Y"] = (
            y_hat.loc[idx] + random_residual
        )

    # Final regression
    X_completed = sm.add_constant(
        completed[["X1", "X2", "X3"]]
    )

    model_final = sm.OLS(
        completed["Y"],
        X_completed
    ).fit()

    bootstrap_beta1.append(
        model_final.params["X1"]
    )

bootstrap_beta1 = np.array(
    bootstrap_beta1
)

# Use the ORIGINAL stochastic point estimate
beta1_stochastic = beta1_hat_mcar_stoch

# Bootstrap inference
se_stochastic = np.std(
    bootstrap_beta1,
    ddof=1
)

ci_lower = np.percentile(
    bootstrap_beta1,
    2.5
)

ci_upper = np.percentile(
    bootstrap_beta1,
    97.5
)

bias_stochastic = beta1_stochastic - beta1_true

print("MCAR Stochastic Regression")
print("beta1 =", beta1_stochastic)
print("SE =", se_stochastic)
print("95% CI =", (ci_lower, ci_upper))
print("Bias =", bias_stochastic)
save_master_result("Stochastic Regression", "MCAR", beta1_stochastic, se_stochastic, ci_lower, ci_upper, bias_stochastic)


MCAR Stochastic Regression
beta1 = 1.7064880364150654
SE = 0.1397469917638977
95% CI = (np.float64(1.487297155555448), np.float64(2.0458637236998345))
Bias = -0.2935119635849346


In [ ]:
# ==========================================
# MAR + STOCHASTIC REGRESSION
# ==========================================

# Make a copy of MAR data
df_mar_stoch = df_mar.copy()

predictors = ["X1", "X2", "X3"]

# Observed and missing rows
observed_mar = df_mar[
    df_mar["Y"].notna()
]

missing_mar = df_mar[
    df_mar["Y"].isna()
]


# ------------------------------------------
# Fit regression using observed Y
# ------------------------------------------

X_mar_obs = sm.add_constant(
    observed_mar[predictors]
)

model_mar_stoch = sm.OLS(
    observed_mar["Y"],
    X_mar_obs
)

results_mar_stoch = (
    model_mar_stoch.fit()
)


# ------------------------------------------
# Obtain observed residuals
# ------------------------------------------

residuals_mar = (
    results_mar_stoch.resid
)


# ------------------------------------------
# Predict missing Y
# ------------------------------------------

X_mar_missing = sm.add_constant(
    missing_mar[predictors]
)

predicted_y_mar = (
    results_mar_stoch.predict(
        X_mar_missing
    )
)


# ------------------------------------------
# Randomly sample residuals
# ------------------------------------------

np.random.seed(42)

random_residuals_mar = np.random.choice(
    residuals_mar,
    size=len(missing_mar),
    replace=True
)


# ------------------------------------------
# Create stochastic imputations
# ------------------------------------------

stochastic_y_mar = (
    predicted_y_mar.to_numpy()
    + random_residuals_mar
)


# ------------------------------------------
# Fill missing values
# ------------------------------------------

df_mar_stoch.loc[
    missing_mar.index,
    "Y"
] = stochastic_y_mar


# ------------------------------------------
# Check
# ------------------------------------------

print(
    "Missing Y after stochastic regression:",
    df_mar_stoch["Y"].isna().sum()
)


# ------------------------------------------
# Regression after imputation
# ------------------------------------------

X_mar_stoch_final = sm.add_constant(
    df_mar_stoch[predictors]
)

results_mar_stoch_final = sm.OLS(
    df_mar_stoch["Y"],
    X_mar_stoch_final
).fit()


# ------------------------------------------
# Results
# ------------------------------------------

beta1_hat_mar_stoch = (
    results_mar_stoch_final.params["X1"]
)

se_beta1_mar_stoch = (
    results_mar_stoch_final.bse["X1"]
)

ci_beta1_mar_stoch = (
    results_mar_stoch_final.conf_int()
    .loc["X1"]
)

bias_mar_stoch = (
    beta1_hat_mar_stoch - beta1_true
)


print("\n===== MAR + STOCHASTIC REGRESSION =====")

print(
    "True beta1:",
    beta1_true
)

print(
    "Estimated beta1:",
    beta1_hat_mar_stoch
)

print(
    "Standard error:",
    se_beta1_mar_stoch
)

print(
    "95% CI:",
    tuple(ci_beta1_mar_stoch)
)

print(
    "Bias:",
    bias_mar_stoch
)

save_point_estimates("Stochastic Regression", "MAR", results_mar_stoch_final.params)


In [560]:
# ============================================
# MAR: STOCHASTIC REGRESSION + BOOTSTRAP
# ============================================

df = df_mar.copy()

B = 500
bootstrap_beta1 = []

np.random.seed(123)

for b in range(B):

    # Resample rows with replacement
    sample_indices = np.random.choice(
        len(df),
        size=len(df),
        replace=True
    )

    bootstrap_sample = df.iloc[
        sample_indices
    ].reset_index(drop=True)

    observed = bootstrap_sample["Y"].notna()
    missing = bootstrap_sample["Y"].isna()

    # Design matrix
    X_boot = sm.add_constant(
        bootstrap_sample[["X1", "X2", "X3"]]
    )

    # Fit regression using observed Y
    model_obs = sm.OLS(
        bootstrap_sample.loc[observed, "Y"],
        X_boot.loc[observed]
    ).fit()

    # Predict Y
    y_hat = model_obs.predict(X_boot)

    # Observed residuals
    residuals = model_obs.resid.to_numpy()

    completed = bootstrap_sample.copy()

    # Stochastic imputation
    for idx in bootstrap_sample.index[missing]:

        random_residual = np.random.choice(
            residuals
        )

        completed.loc[idx, "Y"] = (
            y_hat.loc[idx] + random_residual
        )

    # Final regression
    X_completed = sm.add_constant(
        completed[["X1", "X2", "X3"]]
    )

    model_final = sm.OLS(
        completed["Y"],
        X_completed
    ).fit()

    bootstrap_beta1.append(
        model_final.params["X1"]
    )

bootstrap_beta1 = np.array(
    bootstrap_beta1
)

# Use the ORIGINAL stochastic point estimate
beta1_stochastic = beta1_hat_mar_stoch

# Bootstrap inference
se_stochastic = np.std(
    bootstrap_beta1,
    ddof=1
)

ci_lower = np.percentile(
    bootstrap_beta1,
    2.5
)

ci_upper = np.percentile(
    bootstrap_beta1,
    97.5
)

bias_stochastic = beta1_stochastic - beta1_true

print("MAR Stochastic Regression")
print("beta1 =", beta1_stochastic)
print("SE =", se_stochastic)
print("95% CI =", (ci_lower, ci_upper))
print("Bias =", bias_stochastic)
save_master_result("Stochastic Regression", "MAR", beta1_stochastic, se_stochastic, ci_lower, ci_upper, bias_stochastic)


MAR Stochastic Regression
beta1 = 1.7590838781494003
SE = 0.1474374853512346
95% CI = (np.float64(1.4988676528799199), np.float64(2.064756428026343))
Bias = -0.24091612185059974


In [ ]:
# ==========================================
# MNAR + STOCHASTIC REGRESSION
# ==========================================

# Make a copy of MNAR data
df_mnar_stoch = df_mnar.copy()

predictors = ["X1", "X2", "X3"]

# Observed and missing rows
observed_mnar = df_mnar[
    df_mnar["Y"].notna()
]

missing_mnar = df_mnar[
    df_mnar["Y"].isna()
]


# ------------------------------------------
# Fit regression using observed Y
# ------------------------------------------

X_mnar_obs = sm.add_constant(
    observed_mnar[predictors]
)

model_mnar_stoch = sm.OLS(
    observed_mnar["Y"],
    X_mnar_obs
)

results_mnar_stoch = (
    model_mnar_stoch.fit()
)


# ------------------------------------------
# Obtain residuals
# ------------------------------------------

residuals_mnar = (
    results_mnar_stoch.resid
)


# ------------------------------------------
# Predict missing Y
# ------------------------------------------

X_mnar_missing = sm.add_constant(
    missing_mnar[predictors]
)

predicted_y_mnar = (
    results_mnar_stoch.predict(
        X_mnar_missing
    )
)


# ------------------------------------------
# Randomly sample residuals
# ------------------------------------------

np.random.seed(42)

random_residuals_mnar = np.random.choice(
    residuals_mnar,
    size=len(missing_mnar),
    replace=True
)


# ------------------------------------------
# Create stochastic imputations
# ------------------------------------------

stochastic_y_mnar = (
    predicted_y_mnar.to_numpy()
    + random_residuals_mnar
)


# ------------------------------------------
# Fill missing values
# ------------------------------------------

df_mnar_stoch.loc[
    missing_mnar.index,
    "Y"
] = stochastic_y_mnar


# ------------------------------------------
# Check
# ------------------------------------------

print(
    "Missing Y after stochastic regression:",
    df_mnar_stoch["Y"].isna().sum()
)


# ------------------------------------------
# Regression after imputation
# ------------------------------------------

X_mnar_stoch_final = sm.add_constant(
    df_mnar_stoch[predictors]
)

results_mnar_stoch_final = sm.OLS(
    df_mnar_stoch["Y"],
    X_mnar_stoch_final
).fit()


# ------------------------------------------
# Results
# ------------------------------------------

beta1_hat_mnar_stoch = (
    results_mnar_stoch_final.params["X1"]
)

se_beta1_mnar_stoch = (
    results_mnar_stoch_final.bse["X1"]
)

ci_beta1_mnar_stoch = (
    results_mnar_stoch_final.conf_int()
    .loc["X1"]
)

bias_mnar_stoch = (
    beta1_hat_mnar_stoch - beta1_true
)


print("\n===== MNAR + STOCHASTIC REGRESSION =====")

print(
    "True beta1:",
    beta1_true
)

print(
    "Estimated beta1:",
    beta1_hat_mnar_stoch
)

print(
    "Standard error:",
    se_beta1_mnar_stoch
)

print(
    "95% CI:",
    tuple(ci_beta1_mnar_stoch)
)

print(
    "Bias:",
    bias_mnar_stoch
)

save_point_estimates("Stochastic Regression", "MNAR", results_mnar_stoch_final.params)


In [562]:
# ============================================
# MNAR: STOCHASTIC REGRESSION + BOOTSTRAP
# ============================================

df = df_mnar.copy()

B = 500
bootstrap_beta1 = []

np.random.seed(123)

for b in range(B):

    # Resample rows with replacement
    sample_indices = np.random.choice(
        len(df),
        size=len(df),
        replace=True
    )

    bootstrap_sample = df.iloc[
        sample_indices
    ].reset_index(drop=True)

    observed = bootstrap_sample["Y"].notna()
    missing = bootstrap_sample["Y"].isna()

    # Design matrix
    X_boot = sm.add_constant(
        bootstrap_sample[["X1", "X2", "X3"]]
    )

    # Fit regression using observed Y
    model_obs = sm.OLS(
        bootstrap_sample.loc[observed, "Y"],
        X_boot.loc[observed]
    ).fit()

    # Predict Y
    y_hat = model_obs.predict(X_boot)

    # Observed residuals
    residuals = model_obs.resid.to_numpy()

    completed = bootstrap_sample.copy()

    # Stochastic imputation
    for idx in bootstrap_sample.index[missing]:

        random_residual = np.random.choice(
            residuals
        )

        completed.loc[idx, "Y"] = (
            y_hat.loc[idx] + random_residual
        )

    # Final regression
    X_completed = sm.add_constant(
        completed[["X1", "X2", "X3"]]
    )

    model_final = sm.OLS(
        completed["Y"],
        X_completed
    ).fit()

    bootstrap_beta1.append(
        model_final.params["X1"]
    )

bootstrap_beta1 = np.array(
    bootstrap_beta1
)

# Use the ORIGINAL stochastic point estimate
beta1_stochastic = beta1_hat_mnar_stoch

# Bootstrap inference
se_stochastic = np.std(
    bootstrap_beta1,
    ddof=1
)

ci_lower = np.percentile(
    bootstrap_beta1,
    2.5
)

ci_upper = np.percentile(
    bootstrap_beta1,
    97.5
)

bias_stochastic = beta1_stochastic - beta1_true

print("MNAR Stochastic Regression")
print("beta1 =", beta1_stochastic)
print("SE =", se_stochastic)
print("95% CI =", (ci_lower, ci_upper))
print("Bias =", bias_stochastic)
save_master_result("Stochastic Regression", "MNAR", beta1_stochastic, se_stochastic, ci_lower, ci_upper, bias_stochastic)


MNAR Stochastic Regression
beta1 = 1.7192783252107278
SE = 0.12430619947044783
95% CI = (np.float64(1.5174632961690224), np.float64(1.9869884957103559))
Bias = -0.28072167478927224


In [ ]:
# ==========================================
# MCAR + REGRESSION IMPUTATION
# ==========================================

# Make a copy of MCAR data
df_mcar_reg = df_mcar.copy()

predictors = ["X1", "X2", "X3"]


# ------------------------------------------
# 1. Separate observed and missing rows
# ------------------------------------------

observed_mcar = df_mcar_reg[
    df_mcar_reg["Y"].notna()
]

missing_mcar = df_mcar_reg[
    df_mcar_reg["Y"].isna()
]


# ------------------------------------------
# 2. Fit regression using observed Y
# ------------------------------------------

X_mcar_obs = sm.add_constant(
    observed_mcar[predictors]
)

model_mcar_reg = sm.OLS(
    observed_mcar["Y"],
    X_mcar_obs
)

results_mcar_reg = model_mcar_reg.fit()


# ------------------------------------------
# 3. Predict missing Y
# ------------------------------------------

X_mcar_missing = sm.add_constant(
    missing_mcar[predictors]
)

predicted_y_mcar = results_mcar_reg.predict(
    X_mcar_missing
)


# ------------------------------------------
# 4. Fill missing Y with predictions
# ------------------------------------------

df_mcar_reg.loc[
    missing_mcar.index,
    "Y"
] = predicted_y_mcar


# ------------------------------------------
# 5. Check
# ------------------------------------------

print(
    "Missing Y after regression imputation:",
    df_mcar_reg["Y"].isna().sum()
)


# ------------------------------------------
# 6. Regression after imputation
# ------------------------------------------

X_mcar_reg_final = sm.add_constant(
    df_mcar_reg[predictors]
)

results_mcar_reg_final = sm.OLS(
    df_mcar_reg["Y"],
    X_mcar_reg_final
).fit()


# ------------------------------------------
# 7. Extract beta1, SE, CI and bias
# ------------------------------------------

beta1_hat_mcar_reg = (
    results_mcar_reg_final.params["X1"]
)

se_beta1_mcar_reg = (
    results_mcar_reg_final.bse["X1"]
)

ci_beta1_mcar_reg = (
    results_mcar_reg_final.conf_int()
    .loc["X1"]
)

bias_mcar_reg = (
    beta1_hat_mcar_reg - beta1_true
)


# ------------------------------------------
# 8. Print results
# ------------------------------------------

print("\n===== MCAR + REGRESSION IMPUTATION =====")

print(
    "True beta1:",
    beta1_true
)

print(
    "Estimated beta1:",
    beta1_hat_mcar_reg
)

print(
    "Standard error:",
    se_beta1_mcar_reg
)

print(
    "95% CI:",
    tuple(ci_beta1_mcar_reg)
)

print(
    "Bias:",
    bias_mcar_reg
)
save_point_estimates("Regression Imputation", "MCAR", results_mcar_reg_final.params)


In [564]:
# ============================================
# MCAR: REGRESSION IMPUTATION + BOOTSTRAP
# ============================================

df = df_mcar.copy()

B = 500
bootstrap_beta1 = []

np.random.seed(123)

for b in range(B):

    # Resample rows with replacement
    sample_indices = np.random.choice(
        len(df),
        size=len(df),
        replace=True
    )

    bootstrap_sample = df.iloc[
        sample_indices
    ].reset_index(drop=True)

    observed = bootstrap_sample["Y"].notna()
    missing = bootstrap_sample["Y"].isna()

    # Design matrix
    X_boot = sm.add_constant(
        bootstrap_sample[["X1", "X2", "X3"]]
    )

    # Fit regression using observed Y only
    model_obs = sm.OLS(
        bootstrap_sample.loc[observed, "Y"],
        X_boot.loc[observed]
    ).fit()

    # Predict Y for all observations
    y_hat = model_obs.predict(X_boot)

    # Regression imputation
    completed = bootstrap_sample.copy()

    completed.loc[missing, "Y"] = (
        y_hat.loc[missing]
    )

    # Final regression on completed data
    X_completed = sm.add_constant(
        completed[["X1", "X2", "X3"]]
    )

    model_final = sm.OLS(
        completed["Y"],
        X_completed
    ).fit()

    bootstrap_beta1.append(
        model_final.params["X1"]
    )

bootstrap_beta1 = np.array(
    bootstrap_beta1
)

# --------------------------------------------
# Original regression-imputation estimate
# --------------------------------------------

observed = df["Y"].notna()
missing = df["Y"].isna()

X_original = sm.add_constant(
    df[["X1", "X2", "X3"]]
)

model_obs = sm.OLS(
    df.loc[observed, "Y"],
    X_original.loc[observed]
).fit()

y_hat = model_obs.predict(X_original)

completed_original = df.copy()

completed_original.loc[missing, "Y"] = (
    y_hat.loc[missing]
)

model_final = sm.OLS(
    completed_original["Y"],
    X_original
).fit()

beta1_regression = model_final.params["X1"]

# Bootstrap inference
se_regression = np.std(
    bootstrap_beta1,
    ddof=1
)

ci_lower = np.percentile(
    bootstrap_beta1,
    2.5
)

ci_upper = np.percentile(
    bootstrap_beta1,
    97.5
)

bias_regression = beta1_regression - 2

print("MCAR Regression Imputation")
print("beta1 =", beta1_regression)
print("SE =", se_regression)
print("95% CI =", (ci_lower, ci_upper))
print("Bias =", bias_regression)
save_master_result("Regression Imputation", "MCAR", beta1_regression, se_regression, ci_lower, ci_upper, bias_regression)


MCAR Regression Imputation
beta1 = 1.7847226108996357
SE = 0.13553333590610625
95% CI = (np.float64(1.5272091458782526), np.float64(2.024765145446183))
Bias = -0.2152773891003643


In [ ]:
# ==========================================
# MAR + REGRESSION IMPUTATION
# ==========================================

# Make a copy of MAR data
df_mar_reg = df_mar.copy()

predictors = ["X1", "X2", "X3"]


# ------------------------------------------
# 1. Separate observed and missing rows
# ------------------------------------------

observed_mar = df_mar_reg[
    df_mar_reg["Y"].notna()
]

missing_mar = df_mar_reg[
    df_mar_reg["Y"].isna()
]


# ------------------------------------------
# 2. Fit regression using observed Y
# ------------------------------------------

X_mar_obs = sm.add_constant(
    observed_mar[predictors]
)

model_mar_reg = sm.OLS(
    observed_mar["Y"],
    X_mar_obs
)

results_mar_reg = model_mar_reg.fit()


# ------------------------------------------
# 3. Predict missing Y
# ------------------------------------------

X_mar_missing = sm.add_constant(
    missing_mar[predictors]
)

predicted_y_mar = results_mar_reg.predict(
    X_mar_missing
)


# ------------------------------------------
# 4. Fill missing Y with predictions
# ------------------------------------------

df_mar_reg.loc[
    missing_mar.index,
    "Y"
] = predicted_y_mar


# ------------------------------------------
# 5. Check
# ------------------------------------------

print(
    "Missing Y after regression imputation:",
    df_mar_reg["Y"].isna().sum()
)


# ------------------------------------------
# 6. Regression after imputation
# ------------------------------------------

X_mar_reg_final = sm.add_constant(
    df_mar_reg[predictors]
)

results_mar_reg_final = sm.OLS(
    df_mar_reg["Y"],
    X_mar_reg_final
).fit()


# ------------------------------------------
# 7. Extract beta1, SE, CI and bias
# ------------------------------------------

beta1_hat_mar_reg = (
    results_mar_reg_final.params["X1"]
)

se_beta1_mar_reg = (
    results_mar_reg_final.bse["X1"]
)

ci_beta1_mar_reg = (
    results_mar_reg_final.conf_int()
    .loc["X1"]
)

bias_mar_reg = (
    beta1_hat_mar_reg - beta1_true
)


# ------------------------------------------
# 8. Print results
# ------------------------------------------

print("\n===== MAR + REGRESSION IMPUTATION =====")

print(
    "True beta1:",
    beta1_true
)

print(
    "Estimated beta1:",
    beta1_hat_mar_reg
)

print(
    "Standard error:",
    se_beta1_mar_reg
)

print(
    "95% CI:",
    tuple(ci_beta1_mar_reg)
)

print(
    "Bias:",
    bias_mar_reg
)
save_point_estimates("Regression Imputation", "MAR", results_mar_reg_final.params)


In [566]:
# ============================================
# MAR: REGRESSION IMPUTATION + BOOTSTRAP
# ============================================

df = df_mar.copy()

B = 500
bootstrap_beta1 = []

np.random.seed(123)

for b in range(B):

    # Resample rows with replacement
    sample_indices = np.random.choice(
        len(df),
        size=len(df),
        replace=True
    )

    bootstrap_sample = df.iloc[
        sample_indices
    ].reset_index(drop=True)

    observed = bootstrap_sample["Y"].notna()
    missing = bootstrap_sample["Y"].isna()

    # Design matrix
    X_boot = sm.add_constant(
        bootstrap_sample[["X1", "X2", "X3"]]
    )

    # Fit regression using observed Y only
    model_obs = sm.OLS(
        bootstrap_sample.loc[observed, "Y"],
        X_boot.loc[observed]
    ).fit()

    # Predict Y
    y_hat = model_obs.predict(X_boot)

    # Regression imputation
    completed = bootstrap_sample.copy()

    completed.loc[missing, "Y"] = (
        y_hat.loc[missing]
    )

    # Final regression
    X_completed = sm.add_constant(
        completed[["X1", "X2", "X3"]]
    )

    model_final = sm.OLS(
        completed["Y"],
        X_completed
    ).fit()

    bootstrap_beta1.append(
        model_final.params["X1"]
    )

bootstrap_beta1 = np.array(
    bootstrap_beta1
)

# --------------------------------------------
# Original regression-imputation estimate
# --------------------------------------------

observed = df["Y"].notna()
missing = df["Y"].isna()

X_original = sm.add_constant(
    df[["X1", "X2", "X3"]]
)

model_obs = sm.OLS(
    df.loc[observed, "Y"],
    X_original.loc[observed]
).fit()

y_hat = model_obs.predict(X_original)

completed_original = df.copy()

completed_original.loc[missing, "Y"] = (
    y_hat.loc[missing]
)

model_final = sm.OLS(
    completed_original["Y"],
    X_original
).fit()

beta1_regression = model_final.params["X1"]

# Bootstrap inference
se_regression = np.std(
    bootstrap_beta1,
    ddof=1
)

ci_lower = np.percentile(
    bootstrap_beta1,
    2.5
)

ci_upper = np.percentile(
    bootstrap_beta1,
    97.5
)

bias_regression = beta1_regression - 2

print("MAR Regression Imputation")
print("beta1 =", beta1_regression)
print("SE =", se_regression)
print("95% CI =", (ci_lower, ci_upper))
print("Bias =", bias_regression)
save_master_result("Regression Imputation", "MAR", beta1_regression, se_regression, ci_lower, ci_upper, bias_regression)


MAR Regression Imputation
beta1 = 1.7768964817621877
SE = 0.1360374250155196
95% CI = (np.float64(1.5324465881006826), np.float64(2.0554792572194356))
Bias = -0.22310351823781227


In [ ]:
# ==========================================
# MNAR + REGRESSION IMPUTATION
# ==========================================

# Make a copy of MNAR data
df_mnar_reg = df_mnar.copy()

predictors = ["X1", "X2", "X3"]


# ------------------------------------------
# 1. Separate observed and missing rows
# ------------------------------------------

observed_mnar = df_mnar_reg[
    df_mnar_reg["Y"].notna()
]

missing_mnar = df_mnar_reg[
    df_mnar_reg["Y"].isna()
]


# ------------------------------------------
# 2. Fit regression using observed Y
# ------------------------------------------

X_mnar_obs = sm.add_constant(
    observed_mnar[predictors]
)

model_mnar_reg = sm.OLS(
    observed_mnar["Y"],
    X_mnar_obs
)

results_mnar_reg = model_mnar_reg.fit()


# ------------------------------------------
# 3. Predict missing Y
# ------------------------------------------

X_mnar_missing = sm.add_constant(
    missing_mnar[predictors]
)

predicted_y_mnar = results_mnar_reg.predict(
    X_mnar_missing
)


# ------------------------------------------
# 4. Fill missing Y with predictions
# ------------------------------------------

df_mnar_reg.loc[
    missing_mnar.index,
    "Y"
] = predicted_y_mnar


# ------------------------------------------
# 5. Check
# ------------------------------------------

print(
    "Missing Y after regression imputation:",
    df_mnar_reg["Y"].isna().sum()
)


# ------------------------------------------
# 6. Regression after imputation
# ------------------------------------------

X_mnar_reg_final = sm.add_constant(
    df_mnar_reg[predictors]
)

results_mnar_reg_final = sm.OLS(
    df_mnar_reg["Y"],
    X_mnar_reg_final
).fit()


# ------------------------------------------
# 7. Extract beta1, SE, CI and bias
# ------------------------------------------

beta1_hat_mnar_reg = (
    results_mnar_reg_final.params["X1"]
)

se_beta1_mnar_reg = (
    results_mnar_reg_final.bse["X1"]
)

ci_beta1_mnar_reg = (
    results_mnar_reg_final.conf_int()
    .loc["X1"]
)

bias_mnar_reg = (
    beta1_hat_mnar_reg - beta1_true
)


# ------------------------------------------
# 8. Print results
# ------------------------------------------

print("\n===== MNAR + REGRESSION IMPUTATION =====")

print(
    "True beta1:",
    beta1_true
)

print(
    "Estimated beta1:",
    beta1_hat_mnar_reg
)

print(
    "Standard error:",
    se_beta1_mnar_reg
)

print(
    "95% CI:",
    tuple(ci_beta1_mnar_reg)
)

print(
    "Bias:",
    bias_mnar_reg
)
save_point_estimates("Regression Imputation", "MNAR", results_mnar_reg_final.params)


In [568]:
# ============================================
# MNAR: REGRESSION IMPUTATION + BOOTSTRAP
# ============================================

df = df_mnar.copy()

B = 500
bootstrap_beta1 = []

np.random.seed(123)

for b in range(B):

    # Resample rows with replacement
    sample_indices = np.random.choice(
        len(df),
        size=len(df),
        replace=True
    )

    bootstrap_sample = df.iloc[
        sample_indices
    ].reset_index(drop=True)

    observed = bootstrap_sample["Y"].notna()
    missing = bootstrap_sample["Y"].isna()

    # Design matrix
    X_boot = sm.add_constant(
        bootstrap_sample[["X1", "X2", "X3"]]
    )

    # Fit regression using observed Y only
    model_obs = sm.OLS(
        bootstrap_sample.loc[observed, "Y"],
        X_boot.loc[observed]
    ).fit()

    # Predict Y
    y_hat = model_obs.predict(X_boot)

    # Regression imputation
    completed = bootstrap_sample.copy()

    completed.loc[missing, "Y"] = (
        y_hat.loc[missing]
    )

    # Final regression
    X_completed = sm.add_constant(
        completed[["X1", "X2", "X3"]]
    )

    model_final = sm.OLS(
        completed["Y"],
        X_completed
    ).fit()

    bootstrap_beta1.append(
        model_final.params["X1"]
    )

bootstrap_beta1 = np.array(
    bootstrap_beta1
)

# --------------------------------------------
# Original regression-imputation estimate
# --------------------------------------------

observed = df["Y"].notna()
missing = df["Y"].isna()

X_original = sm.add_constant(
    df[["X1", "X2", "X3"]]
)

model_obs = sm.OLS(
    df.loc[observed, "Y"],
    X_original.loc[observed]
).fit()

y_hat = model_obs.predict(X_original)

completed_original = df.copy()

completed_original.loc[missing, "Y"] = (
    y_hat.loc[missing]
)

model_final = sm.OLS(
    completed_original["Y"],
    X_original
).fit()

beta1_regression = model_final.params["X1"]

# Bootstrap inference
se_regression = np.std(
    bootstrap_beta1,
    ddof=1
)

ci_lower = np.percentile(
    bootstrap_beta1,
    2.5
)

ci_upper = np.percentile(
    bootstrap_beta1,
    97.5
)

bias_regression = beta1_regression - 2

print("MNAR Regression Imputation")
print("beta1 =", beta1_regression)
print("SE =", se_regression)
print("95% CI =", (ci_lower, ci_upper))
print("Bias =", bias_regression)
save_master_result("Regression Imputation", "MNAR", beta1_regression, se_regression, ci_lower, ci_upper, bias_regression)


MNAR Regression Imputation
beta1 = 1.7486858337955535
SE = 0.12195692747359113
95% CI = (np.float64(1.5008796538689386), np.float64(1.9811032792381393))
Bias = -0.25131416620444647


In [ ]:
# ==========================================
# MCAR + MICE USING PREDICTIVE MEAN MATCHING
# ==========================================

predictors = ["X1", "X2", "X3"]

# Number of multiple imputations
m = 5

# Number of MICE cycles
n_iterations = 5

# Number of nearest donors considered in PMM
k = 5


# ------------------------------------------
# Store results from each completed dataset
coef_mice = []
# ------------------------------------------

beta1_mice = []
variance_mice = []


# ------------------------------------------
# Repeat the entire MICE process m times
# ------------------------------------------

for imp_number in range(m):

    # --------------------------------------
    # STEP 0: Start with incomplete MCAR data
    # --------------------------------------

    df_work = df_mcar.copy()

    # Temporary mean fill
    initial_mean_y = df_work["Y"].mean()

    df_work["Y"] = (
        df_work["Y"].fillna(initial_mean_y)
    )


    # Remember which Y values were originally
    # observed and which were originally missing

    observed_index = df_mcar.index[
        df_mcar["Y"].notna()
    ]

    missing_index = df_mcar.index[
        df_mcar["Y"].isna()
    ]


    # --------------------------------------
    # MICE CYCLES
    # --------------------------------------

    for iteration in range(n_iterations):

        # ----------------------------------
        # 1. Fit regression using observed Y
        # ----------------------------------

        X_observed = sm.add_constant(
            df_work.loc[
                observed_index,
                predictors
            ]
        )

        y_observed = df_mcar.loc[
            observed_index,
            "Y"
        ]

        model = sm.OLS(
            y_observed,
            X_observed
        )

        results = model.fit()


        # ----------------------------------
        # 2. Predictive values for ALL rows
        # ----------------------------------

        X_all = sm.add_constant(
            df_work[predictors]
        )

        predicted_all = results.predict(
            X_all
        )


        # ----------------------------------
        # 3. Separate observed predictions
        #    and missing-case predictions
        # ----------------------------------

        predicted_observed = predicted_all.loc[
            observed_index
        ]

        predicted_missing = predicted_all.loc[
            missing_index
        ]


        # ----------------------------------
        # 4. PMM: choose a donor for
        #    each missing observation
        # ----------------------------------

        rng = np.random.default_rng(
            1000 + imp_number
        )

        for idx in missing_index:

            # Prediction for this missing row
            target_prediction = (
                predicted_missing.loc[idx]
            )

            # Distance between this missing
            # prediction and every observed
            # prediction
            distances = (
                predicted_observed
                - target_prediction
            ).abs()

            # Find the k closest observed cases
            nearest_donors = (
                distances
                .sort_values()
                .index[:k]
            )

            # Randomly select ONE donor
            donor_idx = rng.choice(
                nearest_donors
            )

            # Take the donor's ACTUAL observed Y
            donor_y = df_mcar.loc[
                donor_idx,
                "Y"
            ]

            # Impute missing Y with donor's Y
            df_work.loc[
                idx,
                "Y"
            ] = donor_y


    # --------------------------------------
    # This is now one completed dataset
    # --------------------------------------

    df_completed = df_work.copy()


    # --------------------------------------
    # Final substantive regression
    # --------------------------------------

    X_final = sm.add_constant(
        df_completed[predictors]
    )

    results_final = sm.OLS(
        df_completed["Y"],
        X_final
    ).fit()


    # --------------------------------------
    coef_mice.append(
        results_final.params[["const", "X1", "X2", "X3"]].to_numpy()
    )

    # Store beta1 and its variance
    # --------------------------------------

    beta1_mice.append(
        results_final.params["X1"]
    )

    variance_mice.append(
        results_final.bse["X1"] ** 2
    )


coef_pooled = np.mean(np.vstack(coef_mice), axis=0)
save_point_estimates("MICE", "MCAR", pd.Series(coef_pooled, index=["const", "X1", "X2", "X3"]))

# ==========================================
# RUBIN'S RULES
# ==========================================

beta1_mice = np.array(beta1_mice)
variance_mice = np.array(variance_mice)


# Average estimate
beta1_pooled = beta1_mice.mean()


# Within-imputation variance
within_variance = variance_mice.mean()


# Between-imputation variance
between_variance = beta1_mice.var(
    ddof=1
)


# Total variance
total_variance = (
    within_variance
    + (1 + 1 / m) * between_variance
)


# Pooled standard error
se_pooled = np.sqrt(
    total_variance
)


# ------------------------------------------
# Degrees of freedom
# ------------------------------------------

if between_variance == 0:

    df_mice = np.inf

else:

    df_mice = (
        (m - 1)
        * (
            1
            + within_variance
            / (
                (1 + 1 / m)
                * between_variance
            )
        ) ** 2
    )


# ------------------------------------------
# 95% Confidence Interval
# ------------------------------------------

critical_value = t.ppf(
    0.975,
    df_mice
)

ci_lower = (
    beta1_pooled
    - critical_value * se_pooled
)

ci_upper = (
    beta1_pooled
    + critical_value * se_pooled
)


# ------------------------------------------
# Bias
# ------------------------------------------

bias_mice = (
    beta1_pooled - beta1_true
)


# ==========================================
# PRINT RESULTS
# ==========================================

print("\n===== MCAR + MICE (PMM) =====")

print(
    "Beta1 from each imputed dataset:",
    beta1_mice
)

print(
    "Pooled beta1:",
    beta1_pooled
)

print(
    "Within-imputation variance:",
    within_variance
)

print(
    "Between-imputation variance:",
    between_variance
)

print(
    "Pooled standard error:",
    se_pooled
)

print(
    "95% CI:",
    (ci_lower, ci_upper)
)

print(
    "Bias:",
    bias_mice
)
save_master_result("MICE", "MCAR", beta1_pooled, se_pooled, ci_lower, ci_upper, bias_mice)


In [ ]:
# ==========================================
# MAR + MICE USING PMM
# ==========================================

predictors = ["X1", "X2", "X3"]

# Number of completed datasets
m = 5

# Number of MICE cycles
n_iterations = 5

# Number of nearest PMM donors
k = 5

# Store results from each completed dataset
coef_mice_mar = []
beta1_mice_mar = []
variance_mice_mar = []


# ------------------------------------------
# Create m completed datasets
# ------------------------------------------

for imp_number in range(m):

    # Start again from ORIGINAL MAR data
    df_work = df_mar.copy()

    # --------------------------------------
    # STEP 0: Temporary mean fill
    # --------------------------------------

    initial_mean_y = df_work["Y"].mean()

    df_work["Y"] = (
        df_work["Y"].fillna(initial_mean_y)
    )


    # Original observed/missing positions
    observed_index = df_mar.index[
        df_mar["Y"].notna()
    ]

    missing_index = df_mar.index[
        df_mar["Y"].isna()
    ]


    # Random generator for this imputed dataset
    rng = np.random.default_rng(
        2000 + imp_number
    )


    # --------------------------------------
    # MICE cycles
    # --------------------------------------

    for iteration in range(n_iterations):

        # Fit Y model using ONLY originally
        # observed Y values
        X_observed = sm.add_constant(
            df_work.loc[
                observed_index,
                predictors
            ]
        )

        y_observed = df_mar.loc[
            observed_index,
            "Y"
        ]

        results = sm.OLS(
            y_observed,
            X_observed
        ).fit()


        # ----------------------------------
        # Predict Y for all rows
        # ----------------------------------

        X_all = sm.add_constant(
            df_work[predictors]
        )

        predicted_all = results.predict(
            X_all
        )

        predicted_observed = (
            predicted_all.loc[observed_index]
        )

        predicted_missing = (
            predicted_all.loc[missing_index]
        )


        # ----------------------------------
        # PMM
        # ----------------------------------

        for idx in missing_index:

            target_prediction = (
                predicted_missing.loc[idx]
            )

            distances = (
                predicted_observed
                - target_prediction
            ).abs()

            nearest_donors = (
                distances
                .sort_values()
                .index[:k]
            )

            # Randomly choose one suitable donor
            donor_idx = rng.choice(
                nearest_donors
            )

            # Take donor's actual observed Y
            donor_y = df_mar.loc[
                donor_idx,
                "Y"
            ]

            # Replace temporary value
            df_work.loc[
                idx,
                "Y"
            ] = donor_y


    # --------------------------------------
    # One completed dataset
    # --------------------------------------

    df_completed_mar = df_work.copy()


    # --------------------------------------
    # Final regression
    # --------------------------------------

    X_final_mar = sm.add_constant(
        df_completed_mar[predictors]
    )

    results_final_mar = sm.OLS(
        df_completed_mar["Y"],
        X_final_mar
    ).fit()


    coef_mice_mar.append(
        results_final_mar.params[["const", "X1", "X2", "X3"]].to_numpy()
    )

    # Store beta1 and variance
    beta1_mice_mar.append(
        results_final_mar.params["X1"]
    )

    variance_mice_mar.append(
        results_final_mar.bse["X1"] ** 2
    )


coef_pooled = np.mean(np.vstack(coef_mice_mar), axis=0)
save_point_estimates("MICE", "MAR", pd.Series(coef_pooled, index=["const", "X1", "X2", "X3"]))

# ==========================================
# RUBIN'S RULES
# ==========================================

beta1_mice_mar = np.array(
    beta1_mice_mar
)

variance_mice_mar = np.array(
    variance_mice_mar
)


# Average beta1
beta1_pooled_mar = (
    beta1_mice_mar.mean()
)

# Within-imputation variance
within_variance_mar = (
    variance_mice_mar.mean()
)

# Between-imputation variance
between_variance_mar = (
    beta1_mice_mar.var(ddof=1)
)

# Total variance
total_variance_mar = (
    within_variance_mar
    + (1 + 1 / m)
    * between_variance_mar
)

# Pooled SE
se_pooled_mar = np.sqrt(
    total_variance_mar
)


# ------------------------------------------
# Degrees of freedom
# ------------------------------------------

if between_variance_mar == 0:

    df_mice_mar = np.inf

else:

    df_mice_mar = (
        (m - 1)
        * (
            1
            + within_variance_mar
            / (
                (1 + 1 / m)
                * between_variance_mar
            )
        ) ** 2
    )


# ------------------------------------------
# 95% CI
# ------------------------------------------

critical_value_mar = t.ppf(
    0.975,
    df_mice_mar
)

ci_lower_mar = (
    beta1_pooled_mar
    - critical_value_mar
    * se_pooled_mar
)

ci_upper_mar = (
    beta1_pooled_mar
    + critical_value_mar
    * se_pooled_mar
)


# ------------------------------------------
# Bias
# ------------------------------------------

bias_mice_mar = (
    beta1_pooled_mar - beta1_true
)


# ------------------------------------------
# Results
# ------------------------------------------

print("\n===== MAR + MICE (PMM) =====")

print(
    "Beta1 from each imputed dataset:",
    beta1_mice_mar
)

print(
    "Pooled beta1:",
    beta1_pooled_mar
)

print(
    "Within-imputation variance:",
    within_variance_mar
)

print(
    "Between-imputation variance:",
    between_variance_mar
)

print(
    "Pooled standard error:",
    se_pooled_mar
)

print(
    "95% CI:",
    (ci_lower_mar, ci_upper_mar)
)

print(
    "Bias:",
    bias_mice_mar
)
save_master_result("MICE", "MAR", beta1_pooled_mar, se_pooled_mar, ci_lower_mar, ci_upper_mar, bias_mice_mar)


In [ ]:
# ==========================================
# MNAR + MICE USING PMM
# ==========================================

predictors = ["X1", "X2", "X3"]

# Number of completed datasets
m = 5

# Number of MICE cycles
n_iterations = 5

# Number of nearest PMM donors
k = 5

# Store results from each completed dataset
coef_mice_mnar = []
beta1_mice_mnar = []
variance_mice_mnar = []


# ------------------------------------------
# Create m completed datasets
# ------------------------------------------

for imp_number in range(m):

    # Start again from ORIGINAL MNAR data
    df_work = df_mnar.copy()

    # --------------------------------------
    # STEP 0: Temporary mean fill
    # --------------------------------------

    initial_mean_y = df_work["Y"].mean()

    df_work["Y"] = (
        df_work["Y"].fillna(initial_mean_y)
    )


    # Original observed/missing positions
    observed_index = df_mnar.index[
        df_mnar["Y"].notna()
    ]

    missing_index = df_mnar.index[
        df_mnar["Y"].isna()
    ]


    # Random generator for this dataset
    rng = np.random.default_rng(
        3000 + imp_number
    )


    # --------------------------------------
    # MICE cycles
    # --------------------------------------

    for iteration in range(n_iterations):

        # Fit Y model using ONLY originally
        # observed Y values
        X_observed = sm.add_constant(
            df_work.loc[
                observed_index,
                predictors
            ]
        )

        y_observed = df_mnar.loc[
            observed_index,
            "Y"
        ]

        results = sm.OLS(
            y_observed,
            X_observed
        ).fit()


        # ----------------------------------
        # Predict Y for all rows
        # ----------------------------------

        X_all = sm.add_constant(
            df_work[predictors]
        )

        predicted_all = results.predict(
            X_all
        )

        predicted_observed = (
            predicted_all.loc[observed_index]
        )

        predicted_missing = (
            predicted_all.loc[missing_index]
        )


        # ----------------------------------
        # PMM
        # ----------------------------------

        for idx in missing_index:

            target_prediction = (
                predicted_missing.loc[idx]
            )

            distances = (
                predicted_observed
                - target_prediction
            ).abs()

            nearest_donors = (
                distances
                .sort_values()
                .index[:k]
            )

            # Randomly choose one suitable donor
            donor_idx = rng.choice(
                nearest_donors
            )

            # Donor's actual observed Y
            donor_y = df_mnar.loc[
                donor_idx,
                "Y"
            ]

            # Replace temporary value
            df_work.loc[
                idx,
                "Y"
            ] = donor_y


    # --------------------------------------
    # One completed dataset
    # --------------------------------------

    df_completed_mnar = df_work.copy()


    # --------------------------------------
    # Final regression
    # --------------------------------------

    X_final_mnar = sm.add_constant(
        df_completed_mnar[predictors]
    )

    results_final_mnar = sm.OLS(
        df_completed_mnar["Y"],
        X_final_mnar
    ).fit()


    coef_mice_mnar.append(
        results_final_mnar.params[["const", "X1", "X2", "X3"]].to_numpy()
    )

    # Store beta1 and variance
    beta1_mice_mnar.append(
        results_final_mnar.params["X1"]
    )

    variance_mice_mnar.append(
        results_final_mnar.bse["X1"] ** 2
    )


coef_pooled = np.mean(np.vstack(coef_mice_mnar), axis=0)
save_point_estimates("MICE", "MNAR", pd.Series(coef_pooled, index=["const", "X1", "X2", "X3"]))

# ==========================================
# RUBIN'S RULES
# ==========================================

beta1_mice_mnar = np.array(
    beta1_mice_mnar
)

variance_mice_mnar = np.array(
    variance_mice_mnar
)


# Average beta1
beta1_pooled_mnar = (
    beta1_mice_mnar.mean()
)

# Within-imputation variance
within_variance_mnar = (
    variance_mice_mnar.mean()
)

# Between-imputation variance
between_variance_mnar = (
    beta1_mice_mnar.var(ddof=1)
)

# Total variance
total_variance_mnar = (
    within_variance_mnar
    + (1 + 1 / m)
    * between_variance_mnar
)

# Pooled SE
se_pooled_mnar = np.sqrt(
    total_variance_mnar
)


# ------------------------------------------
# Degrees of freedom
# ------------------------------------------

if between_variance_mnar == 0:

    df_mice_mnar = np.inf

else:

    df_mice_mnar = (
        (m - 1)
        * (
            1
            + within_variance_mnar
            / (
                (1 + 1 / m)
                * between_variance_mnar
            )
        ) ** 2
    )


# ------------------------------------------
# 95% CI
# ------------------------------------------

critical_value_mnar = t.ppf(
    0.975,
    df_mice_mnar
)

ci_lower_mnar = (
    beta1_pooled_mnar
    - critical_value_mnar
    * se_pooled_mnar
)

ci_upper_mnar = (
    beta1_pooled_mnar
    + critical_value_mnar
    * se_pooled_mnar
)


# ------------------------------------------
# Bias
# ------------------------------------------

bias_mice_mnar = (
    beta1_pooled_mnar - beta1_true
)


# ------------------------------------------
# Results
# ------------------------------------------

print("\n===== MNAR + MICE (PMM) =====")

print(
    "Beta1 from each imputed dataset:",
    beta1_mice_mnar
)

print(
    "Pooled beta1:",
    beta1_pooled_mnar
)

print(
    "Within-imputation variance:",
    within_variance_mnar
)

print(
    "Between-imputation variance:",
    between_variance_mnar
)

print(
    "Pooled standard error:",
    se_pooled_mnar
)

print(
    "95% CI:",
    (ci_lower_mnar, ci_upper_mnar)
)

print(
    "Bias:",
    bias_mice_mnar
)
save_master_result("MICE", "MNAR", beta1_pooled_mnar, se_pooled_mnar, ci_lower_mnar, ci_upper_mnar, bias_mice_mnar)


In [ ]:
# =========================
# EM FOR MCAR DATA
# =========================

df = df_mcar.copy()

# Identify observed and missing Y
observed = df["Y"].notna().to_numpy()
missing = ~observed

# Design matrix: 1, X1, X2, X3
X = sm.add_constant(
    df[["X1", "X2", "X3"]].to_numpy()
)

# -------------------------
# INITIAL VALUES
# -------------------------

# Deliberately poor starting values
beta = np.zeros(4)

# Start sigma^2 with the variance of the observed Y values
y_obs = df.loc[observed, "Y"].to_numpy()
sigma2 = np.var(y_obs)

# Store estimates from every iteration
history = []

# -------------------------
# EM ALGORITHM
# -------------------------

max_iter = 100
tol = 1e-8

for iteration in range(max_iter):

    # =========================
    # E-STEP
    # =========================

    # Current expected Y
    y_hat = X @ beta

    # E[Y]
    Ey = df["Y"].to_numpy().copy()
    Ey[missing] = y_hat[missing]

    # E[Y^2]
    Ey2 = df["Y"].to_numpy() ** 2
    Ey2[missing] = y_hat[missing] ** 2 + sigma2

    # =========================
    # M-STEP
    # =========================

    # Update beta
    beta_new = np.linalg.solve(
        X.T @ X,
        X.T @ Ey
    )

    # Update sigma^2
    fitted = X @ beta_new

    expected_squared_residual = (
        Ey2
        - 2 * fitted * Ey
        + fitted ** 2
    )

    sigma2_new = np.mean(expected_squared_residual)

    # Save this iteration
    history.append([
        iteration + 1,
        beta_new[0],
        beta_new[1],
        beta_new[2],
        beta_new[3],
        sigma2_new
    ])

    # =========================
    # CONVERGENCE CHECK
    # =========================

    beta_change = np.max(np.abs(beta_new - beta))
    sigma_change = abs(sigma2_new - sigma2)

    beta = beta_new
    sigma2 = sigma2_new

    if beta_change < tol and sigma_change < tol:
        break

# -------------------------
# RESULTS
# -------------------------

history_df = pd.DataFrame(
    history,
    columns=[
        "Iteration",
        "beta0",
        "beta1",
        "beta2",
        "beta3",
        "sigma2"
    ]
)

print("MCAR EM converged after", iteration + 1, "iterations")
print("\nFinal estimates:")
print("beta0 =", beta[0])
print("beta1 =", beta[1])
print("beta2 =", beta[2])
print("beta3 =", beta[3])
print("sigma^2 =", sigma2)

print("\nIteration history:")
print(history_df.to_string(index=False))
save_point_estimates("EM", "MCAR", pd.Series(beta, index=["const", "X1", "X2", "X3"]))


In [ ]:
# =========================
# EM FOR MAR DATA
# =========================

df = df_mar.copy()

# Identify observed and missing Y
observed = df["Y"].notna().to_numpy()
missing = ~observed

# Design matrix: 1, X1, X2, X3
X = sm.add_constant(
    df[["X1", "X2", "X3"]].to_numpy()
)

# -------------------------
# INITIAL VALUES
# -------------------------

beta = np.zeros(4)

y_obs = df.loc[observed, "Y"].to_numpy()
sigma2 = np.var(y_obs)

history = []

# -------------------------
# EM ALGORITHM
# -------------------------

max_iter = 100
tol = 1e-8

for iteration in range(max_iter):

    # =========================
    # E-STEP
    # =========================

    y_hat = X @ beta

    # E[Y]
    Ey = df["Y"].to_numpy().copy()
    Ey[missing] = y_hat[missing]

    # E[Y^2]
    Ey2 = df["Y"].to_numpy() ** 2
    Ey2[missing] = y_hat[missing] ** 2 + sigma2

    # =========================
    # M-STEP
    # =========================

    beta_new = np.linalg.solve(
        X.T @ X,
        X.T @ Ey
    )

    fitted = X @ beta_new

    expected_squared_residual = (
        Ey2
        - 2 * fitted * Ey
        + fitted ** 2
    )

    sigma2_new = np.mean(expected_squared_residual)

    history.append([
        iteration + 1,
        beta_new[0],
        beta_new[1],
        beta_new[2],
        beta_new[3],
        sigma2_new
    ])

    # =========================
    # CONVERGENCE CHECK
    # =========================

    beta_change = np.max(np.abs(beta_new - beta))
    sigma_change = abs(sigma2_new - sigma2)

    beta = beta_new
    sigma2 = sigma2_new

    if beta_change < tol and sigma_change < tol:
        break

# -------------------------
# RESULTS
# -------------------------

history_df = pd.DataFrame(
    history,
    columns=[
        "Iteration",
        "beta0",
        "beta1",
        "beta2",
        "beta3",
        "sigma2"
    ]
)

print("MAR EM converged after", iteration + 1, "iterations")
print("\nFinal estimates:")
print("beta0 =", beta[0])
print("beta1 =", beta[1])
print("beta2 =", beta[2])
print("beta3 =", beta[3])
print("sigma^2 =", sigma2)

print("\nIteration history:")
print(history_df.to_string(index=False))
save_point_estimates("EM", "MAR", pd.Series(beta, index=["const", "X1", "X2", "X3"]))


In [ ]:
# =========================
# EM FOR MNAR DATA
# =========================

df = df_mnar.copy()

# Identify observed and missing Y
observed = df["Y"].notna().to_numpy()
missing = ~observed

# Design matrix: 1, X1, X2, X3
X = sm.add_constant(
    df[["X1", "X2", "X3"]].to_numpy()
)

# -------------------------
# INITIAL VALUES
# -------------------------

beta = np.zeros(4)

y_obs = df.loc[observed, "Y"].to_numpy()
sigma2 = np.var(y_obs)

history = []

# -------------------------
# EM ALGORITHM
# -------------------------

max_iter = 100
tol = 1e-8

for iteration in range(max_iter):

    # =========================
    # E-STEP
    # =========================

    y_hat = X @ beta

    # E[Y]
    Ey = df["Y"].to_numpy().copy()
    Ey[missing] = y_hat[missing]

    # E[Y^2]
    Ey2 = df["Y"].to_numpy() ** 2
    Ey2[missing] = y_hat[missing] ** 2 + sigma2

    # =========================
    # M-STEP
    # =========================

    beta_new = np.linalg.solve(
        X.T @ X,
        X.T @ Ey
    )

    fitted = X @ beta_new

    expected_squared_residual = (
        Ey2
        - 2 * fitted * Ey
        + fitted ** 2
    )

    sigma2_new = np.mean(expected_squared_residual)

    history.append([
        iteration + 1,
        beta_new[0],
        beta_new[1],
        beta_new[2],
        beta_new[3],
        sigma2_new
    ])

    # =========================
    # CONVERGENCE CHECK
    # =========================

    beta_change = np.max(np.abs(beta_new - beta))
    sigma_change = abs(sigma2_new - sigma2)

    beta = beta_new
    sigma2 = sigma2_new

    if beta_change < tol and sigma_change < tol:
        break

# -------------------------
# RESULTS
# -------------------------

history_df = pd.DataFrame(
    history,
    columns=[
        "Iteration",
        "beta0",
        "beta1",
        "beta2",
        "beta3",
        "sigma2"
    ]
)

print("MNAR EM converged after", iteration + 1, "iterations")
print("\nFinal estimates:")
print("beta0 =", beta[0])
print("beta1 =", beta[1])
print("beta2 =", beta[2])
print("beta3 =", beta[3])
print("sigma^2 =", sigma2)

print("\nIteration history:")
print(history_df.to_string(index=False))
save_point_estimates("EM", "MNAR", pd.Series(beta, index=["const", "X1", "X2", "X3"]))


In [575]:
# ============================================
# MCAR: EM + BOOTSTRAP SE, CI AND BIAS
# ============================================

df = df_mcar.copy()

def em_fit(data, max_iter=100, tol=1e-8):

    observed = data["Y"].notna().to_numpy()
    missing = ~observed

    X = sm.add_constant(
        data[["X1", "X2", "X3"]].to_numpy()
    )

    # Starting values
    beta = np.zeros(4)
    y_obs = data.loc[observed, "Y"].to_numpy()
    sigma2 = np.var(y_obs)

    # EM
    for iteration in range(max_iter):

        # E-step
        y_hat = X @ beta

        Ey = data["Y"].to_numpy().copy()
        Ey[missing] = y_hat[missing]

        Ey2 = data["Y"].to_numpy() ** 2
        Ey2[missing] = y_hat[missing] ** 2 + sigma2

        # M-step
        beta_new = np.linalg.solve(
            X.T @ X,
            X.T @ Ey
        )

        fitted = X @ beta_new

        expected_squared_residual = (
            Ey2
            - 2 * fitted * Ey
            + fitted ** 2
        )

        sigma2_new = np.mean(expected_squared_residual)

        # Convergence
        beta_change = np.max(np.abs(beta_new - beta))
        sigma_change = abs(sigma2_new - sigma2)

        beta = beta_new
        sigma2 = sigma2_new

        if beta_change < tol and sigma_change < tol:
            break

    return beta, sigma2


# Original EM estimate
beta_em, sigma2_em = em_fit(df)

beta1_em = beta_em[1]

# --------------------------------------------
# BOOTSTRAP
# --------------------------------------------

B = 500
bootstrap_beta1 = []

np.random.seed(123)

for b in range(B):

    # Resample rows with replacement
    sample_indices = np.random.choice(
        len(df),
        size=len(df),
        replace=True
    )

    bootstrap_sample = df.iloc[sample_indices].reset_index(drop=True)

    # Fit EM to bootstrap sample
    beta_boot, _ = em_fit(bootstrap_sample)

    bootstrap_beta1.append(beta_boot[1])


bootstrap_beta1 = np.array(bootstrap_beta1)

# Bootstrap standard error
se_em = np.std(bootstrap_beta1, ddof=1)

# Percentile 95% CI
ci_lower = np.percentile(bootstrap_beta1, 2.5)
ci_upper = np.percentile(bootstrap_beta1, 97.5)

# Bias relative to true beta1 = 2
bias_em = beta1_em - 2

print("MCAR EM")
print("beta1 =", beta1_em)
print("SE =", se_em)
print("95% CI =", (ci_lower, ci_upper))
print("Bias =", bias_em)
save_master_result("EM", "MCAR", beta1_em, se_em, ci_lower, ci_upper, bias_em)


MCAR EM
beta1 = 1.7847226056536811
SE = 0.1355333357531633
95% CI = (np.float64(1.527209139943724), np.float64(2.024765142871744))
Bias = -0.21527739434631887


In [576]:
# ============================================
# MAR: EM + BOOTSTRAP SE, CI AND BIAS
# ============================================

df = df_mar.copy()

def em_fit(data, max_iter=100, tol=1e-8):

    observed = data["Y"].notna().to_numpy()
    missing = ~observed

    X = sm.add_constant(
        data[["X1", "X2", "X3"]].to_numpy()
    )

    beta = np.zeros(4)

    y_obs = data.loc[observed, "Y"].to_numpy()
    sigma2 = np.var(y_obs)

    for iteration in range(max_iter):

        # E-step
        y_hat = X @ beta

        Ey = data["Y"].to_numpy().copy()
        Ey[missing] = y_hat[missing]

        Ey2 = data["Y"].to_numpy() ** 2
        Ey2[missing] = y_hat[missing] ** 2 + sigma2

        # M-step
        beta_new = np.linalg.solve(
            X.T @ X,
            X.T @ Ey
        )

        fitted = X @ beta_new

        expected_squared_residual = (
            Ey2
            - 2 * fitted * Ey
            + fitted ** 2
        )

        sigma2_new = np.mean(expected_squared_residual)

        beta_change = np.max(np.abs(beta_new - beta))
        sigma_change = abs(sigma2_new - sigma2)

        beta = beta_new
        sigma2 = sigma2_new

        if beta_change < tol and sigma_change < tol:
            break

    return beta, sigma2


# Original EM estimate
beta_em, sigma2_em = em_fit(df)

beta1_em = beta_em[1]

# --------------------------------------------
# BOOTSTRAP
# --------------------------------------------

B = 500
bootstrap_beta1 = []

np.random.seed(123)

for b in range(B):

    sample_indices = np.random.choice(
        len(df),
        size=len(df),
        replace=True
    )

    bootstrap_sample = df.iloc[sample_indices].reset_index(drop=True)

    beta_boot, _ = em_fit(bootstrap_sample)

    bootstrap_beta1.append(beta_boot[1])


bootstrap_beta1 = np.array(bootstrap_beta1)

# Inference
se_em = np.std(bootstrap_beta1, ddof=1)

ci_lower = np.percentile(
    bootstrap_beta1,
    2.5
)

ci_upper = np.percentile(
    bootstrap_beta1,
    97.5
)

bias_em = beta1_em - 2

print("MAR EM")
print("beta1 =", beta1_em)
print("SE =", se_em)
print("95% CI =", (ci_lower, ci_upper))
print("Bias =", bias_em)
save_master_result("EM", "MAR", beta1_em, se_em, ci_lower, ci_upper, bias_em)


MAR EM
beta1 = 1.7768964855201035
SE = 0.13603742519119352
95% CI = (np.float64(1.5324465911255774), np.float64(2.055479267137561))
Bias = -0.22310351447989651


In [577]:
# ============================================
# MNAR: EM + BOOTSTRAP SE, CI AND BIAS
# ============================================

df = df_mnar.copy()

def em_fit(data, max_iter=100, tol=1e-8):

    observed = data["Y"].notna().to_numpy()
    missing = ~observed

    X = sm.add_constant(
        data[["X1", "X2", "X3"]].to_numpy()
    )

    beta = np.zeros(4)

    y_obs = data.loc[observed, "Y"].to_numpy()
    sigma2 = np.var(y_obs)

    for iteration in range(max_iter):

        # E-step
        y_hat = X @ beta

        Ey = data["Y"].to_numpy().copy()
        Ey[missing] = y_hat[missing]

        Ey2 = data["Y"].to_numpy() ** 2
        Ey2[missing] = y_hat[missing] ** 2 + sigma2

        # M-step
        beta_new = np.linalg.solve(
            X.T @ X,
            X.T @ Ey
        )

        fitted = X @ beta_new

        expected_squared_residual = (
            Ey2
            - 2 * fitted * Ey
            + fitted ** 2
        )

        sigma2_new = np.mean(expected_squared_residual)

        beta_change = np.max(np.abs(beta_new - beta))
        sigma_change = abs(sigma2_new - sigma2)

        beta = beta_new
        sigma2 = sigma2_new

        if beta_change < tol and sigma_change < tol:
            break

    return beta, sigma2


# Original EM estimate
beta_em, sigma2_em = em_fit(df)

beta1_em = beta_em[1]

# --------------------------------------------
# BOOTSTRAP
# --------------------------------------------

B = 500
bootstrap_beta1 = []

np.random.seed(123)

for b in range(B):

    sample_indices = np.random.choice(
        len(df),
        size=len(df),
        replace=True
    )

    bootstrap_sample = df.iloc[sample_indices].reset_index(drop=True)

    beta_boot, _ = em_fit(bootstrap_sample)

    bootstrap_beta1.append(beta_boot[1])


bootstrap_beta1 = np.array(bootstrap_beta1)

# Inference
se_em = np.std(bootstrap_beta1, ddof=1)

ci_lower = np.percentile(
    bootstrap_beta1,
    2.5
)

ci_upper = np.percentile(
    bootstrap_beta1,
    97.5
)

bias_em = beta1_em - 2

print("MNAR EM")
print("beta1 =", beta1_em)
print("SE =", se_em)
print("95% CI =", (ci_lower, ci_upper))
print("Bias =", bias_em)
save_master_result("EM", "MNAR", beta1_em, se_em, ci_lower, ci_upper, bias_em)


MNAR EM
beta1 = 1.7486858331420063
SE = 0.12195692744949976
95% CI = (np.float64(1.5008796521707792), np.float64(1.9811032790562093))
Bias = -0.2513141668579937


# Final Comparison — Methods Within Each Missingness Mechanism

The main comparison in this project is **between missing-data handling methods within a given missingness mechanism**.

MCAR, MAR and MNAR are treated as three experimental settings. The purpose is not to declare one missingness mechanism "better" than another, because the mechanism represents how the missingness arose and is not something the analyst chooses.

Therefore, the final results are organized as:

\[
	ext{MCAR} ightarrow 	ext{compare all methods}
\]

\[
	ext{MAR} ightarrow 	ext{compare all methods}
\]

\[
	ext{MNAR} ightarrow 	ext{compare all methods}.
\]

This structure keeps the comparison aligned with the main question: **given a particular missingness mechanism, how do the available handling methods differ in their estimates and inference?**


In [ ]:
# ==========================================
# MASTER RESULTS — ORGANIZED BY MECHANISM
# ==========================================

import pandas as pd
from IPython.display import display, Markdown

methods = [
    "Complete Case",
    "Mean Imputation",
    "Hot-Deck",
    "Regression Imputation",
    "Stochastic Regression",
    "MICE",
    "EM"
]

mechanisms = ["MCAR", "MAR", "MNAR"]

# Build the two result tables from the values
# produced by the method sections.
all_coefficient_results = pd.DataFrame(
    point_estimate_store.values()
)

master_results = pd.DataFrame(
    master_result_store.values()
)

for table in [all_coefficient_results, master_results]:
    table["Method"] = table["Method"].astype(str)
    table["Mechanism"] = table["Mechanism"].astype(str)

all_coefficient_results = (
    all_coefficient_results[
        all_coefficient_results["Method"].isin(methods)
        & all_coefficient_results["Mechanism"].isin(mechanisms)
    ]
)

master_results = (
    master_results[
        master_results["Method"].isin(methods)
        & master_results["Mechanism"].isin(mechanisms)
    ]
)

print("True coefficient values:")
print("Beta0 = 5, Beta1 = 2, Beta2 = -1.5, Beta3 = 1")

for mechanism in mechanisms:

    coef_table = (
        all_coefficient_results[
            all_coefficient_results["Mechanism"] == mechanism
        ]
        .set_index("Method")
        .reindex(methods)
        .reset_index()
    )

    inference_table = (
        master_results[
            master_results["Mechanism"] == mechanism
        ]
        .set_index("Method")
        .reindex(methods)
        .reset_index()
    )

    final_table = coef_table[
        ["Method", "Beta0", "Beta1", "Beta2", "Beta3"]
    ].merge(
        inference_table[
            ["Method", "SE", "CI_Lower", "CI_Upper", "Bias"]
        ],
        on="Method",
        how="left"
    )

    final_table["Absolute_Bias"] = final_table["Bias"].abs()

    final_table["95% CI for Beta1"] = (
        final_table["CI_Lower"].map(lambda x: f"{x:.3f}")
        + ", "
        + final_table["CI_Upper"].map(lambda x: f"{x:.3f}")
    )

    final_table = final_table[
        [
            "Method",
            "Beta0",
            "Beta1",
            "Beta2",
            "Beta3",
            "SE",
            "95% CI for Beta1",
            "Bias",
            "Absolute_Bias"
        ]
    ].copy()

    numeric_columns = [
        "Beta0", "Beta1", "Beta2", "Beta3",
        "SE", "Bias", "Absolute_Bias"
    ]

    final_table[numeric_columns] = (
        final_table[numeric_columns].round(3)
    )

    display(Markdown(f"## {mechanism}: Compare the Methods"))
    display(final_table)


# Visualisations

The main visual comparison is organized **within each missingness mechanism**:

- MCAR → compare all seven methods
- MAR → compare all seven methods
- MNAR → compare all seven methods

For each mechanism, the plots show the estimated β₁ with its 95% confidence interval and the absolute bias.

An additional overview plot reverses the axes so that **MCAR, MAR and MNAR appear on the x-axis**, with the seven methods grouped within each mechanism. This provides a compact visual summary without changing the main method-within-mechanism interpretation.


In [ ]:
# ==========================================
# PLOT 1 — BETA1 + 95% CI, ONE MECHANISM AT A TIME
# ==========================================

import matplotlib.pyplot as plt
import numpy as np

for mechanism in mechanisms:

    subset = (
        master_results[
            master_results["Mechanism"] == mechanism
        ]
        .set_index("Method")
        .reindex(methods)
    )

    x = np.arange(len(methods))

    lower_error = (
        subset["Beta1"].to_numpy()
        - subset["CI_Lower"].to_numpy()
    )

    upper_error = (
        subset["CI_Upper"].to_numpy()
        - subset["Beta1"].to_numpy()
    )

    plt.figure(figsize=(11, 5.5))

    plt.errorbar(
        x,
        subset["Beta1"].to_numpy(),
        yerr=np.vstack([
            lower_error,
            upper_error
        ]),
        fmt="o",
        capsize=5
    )

    plt.axhline(
        2,
        linestyle="--",
        label="True Beta1 = 2"
    )

    plt.xticks(
        x,
        methods,
        rotation=30,
        ha="right"
    )

    plt.ylabel("Beta1 estimate")
    plt.xlabel("Method")
    plt.title(
        f"{mechanism}: Comparison of Missing-Data Handling Methods"
    )
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# ==========================================
# PLOT 2 — ABSOLUTE BIAS, ONE MECHANISM AT A TIME
# ==========================================

for mechanism in mechanisms:

    subset = (
        master_results[
            master_results["Mechanism"] == mechanism
        ]
        .set_index("Method")
        .reindex(methods)
    )

    x = np.arange(len(methods))

    plt.figure(figsize=(11, 5.5))

    plt.bar(
        x,
        subset["Bias"].abs().to_numpy()
    )

    plt.xticks(
        x,
        methods,
        rotation=30,
        ha="right"
    )

    plt.ylabel("Absolute bias of Beta1")
    plt.xlabel("Method")
    plt.title(
        f"{mechanism}: Absolute Bias Across Methods"
    )

    plt.tight_layout()
    plt.show()


## Additional Overview Plot — Mechanisms on the X-axis

The mechanism-specific plots above are the main analytical comparison. This additional plot gives a compact overview with **MCAR, MAR and MNAR on the x-axis**, and the seven methods grouped within each mechanism.

This is the orientation of the plot used in the original comparison you liked.


In [ ]:
# ==========================================
# PLOT 3 — MECHANISM ON X-AXIS
# GROUPED BAR CHART OF BETA1
# ==========================================

import matplotlib.pyplot as plt
import numpy as np

x = np.arange(len(mechanisms))
n_methods = len(methods)
width = 0.11

plt.figure(figsize=(12, 6))

for i, method in enumerate(methods):

    subset = (
        master_results[
            master_results["Method"] == method
        ]
        .set_index("Mechanism")
        .reindex(mechanisms)
    )

    offset = (
        i - (n_methods - 1) / 2
    ) * width

    plt.bar(
        x + offset,
        subset["Beta1"].to_numpy(),
        width=width,
        label=method
    )

plt.axhline(
    beta1_true,
    linestyle="--",
    label="True Beta1 = 2"
)

plt.xticks(
    x,
    mechanisms
)

plt.xlabel("Missingness mechanism")
plt.ylabel("Beta1 estimate")
plt.title(
    "Beta1 Estimates by Method and Missingness Mechanism"
)

plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()


# 7. Results and Interpretation

The interpretation is organized **within each missingness mechanism**. MCAR, MAR and MNAR are treated as conditions under which the same seven handling approaches are evaluated.

The primary parameter is \(\beta_1\), whose true value is 2.


## 7.1 MCAR — Comparison of Methods

Under MCAR, the estimated \(\beta_1\) values ranged from approximately 1.089 to 1.785.

Mean imputation produced the largest deviation from the true value:

\[
\hat\beta_1=1.089,\qquad
Bias=-0.911.
\]

Its bootstrap standard error was approximately 0.229 and its 95% confidence interval was

\[
(0.649,\ 1.565),
\]

which lies entirely below the true value of 2.

The other methods produced estimates considerably closer to 2. Complete-case analysis, regression imputation and EM all gave approximately

\[
\hat\beta_1=1.785.
\]

Hot-deck gave 1.658, stochastic regression gave 1.706, and MICE gave a pooled estimate of 1.659.

Thus, within the MCAR setting, the main distinction observed in this simulation is between the strong attenuation produced by mean imputation and the smaller deviations produced by the other approaches.

The confidence intervals also differ. The complete-case, regression-imputation and EM intervals extend above 2, whereas the mean-imputation interval remains well below it.


## 7.2 MAR — Comparison of Methods

Under MAR, missingness was related to the observed predictor \(X_1\), so the methods were compared under the same MAR condition.

Mean imputation again produced a substantial downward shift:

\[
\hat\beta_1=1.307,
\]

with

\[
Bias=-0.693.
\]

Its 95% confidence interval was

\[
(0.915,\ 1.659),
\]

which is below the true value of 2.

The other methods were much closer to the true coefficient. Hot-deck produced

\[
\hat\beta_1=1.787,
\]

stochastic regression produced 1.759, and MICE produced the pooled estimate

\[
1.797.
\]

Complete-case analysis, regression imputation and EM produced approximately 1.777.

The MICE estimate is particularly useful to interpret here because the imputation procedure explicitly used the observed predictors to construct plausible donors. The five completed datasets produced \(\beta_1\) estimates ranging from approximately 1.768 to 1.830, giving a pooled value of 1.797.

The MAR comparison therefore shows how substantially the handling method can change both the estimated regression effect and the associated interval even when the missingness mechanism itself is held fixed.


## 7.3 MNAR — Comparison of Methods

Under MNAR, missingness was deliberately made dependent on the response variable \(Y\) itself.

Mean imputation produced

\[
\hat\beta_1=1.413
\]

with bias

\[
-0.587.
\]

Its 95% confidence interval was

\[
(1.047,\ 1.702).
\]

The other approaches produced estimates between approximately 1.659 and 1.785, with MICE producing a pooled estimate of 1.722.

For the MNAR case, however, the comparison needs a stronger qualification than in MCAR or MAR. Because the missingness mechanism depends on the unobserved outcome itself, methods based on MAR-type assumptions do not automatically solve the missingness problem. The resulting numbers are therefore best interpreted as the behavior of the implemented procedures under a deliberately difficult MNAR stress test, not as evidence that any of the methods identifies the true MNAR mechanism.

The confidence-interval plot makes the practical consequence visible: in this particular simulation, all seven intervals for \(\beta_1\) remain below the true value of 2.


## 7.4 Cross-Method Interpretation

The central comparison is therefore **method within mechanism**, not mechanism against mechanism.

For each of MCAR, MAR and MNAR, the same seven approaches were applied to the corresponding incomplete dataset and their estimates, uncertainty and bias were compared.

One consistent feature of the simulation is the strong downward distortion associated with mean imputation. This is consistent with its theoretical tendency to reduce outcome variation when missing values are replaced by a common mean.

Hot-deck and stochastic regression preserve more variation in the imputed outcome and consequently produce different regression estimates from simple mean replacement.

MICE differs from the single-imputation approaches because it creates multiple completed datasets and incorporates both within-imputation and between-imputation variability through Rubin's rules.

EM differs conceptually again because it estimates the model from incomplete data rather than treating a single completed dataset as the final analysis dataset.

The numerical results should be treated as findings from this controlled simulation. They illustrate how the handling method can change the statistical analysis under a specified missingness condition; they do not establish a universal ranking of the methods.


# 8. Limitations

This analysis uses one simulated complete dataset of 100 observations and one realized missingness pattern for each of MCAR, MAR and MNAR. The project therefore demonstrates the methods in a controlled setting rather than estimating their long-run performance across many independently generated datasets.

The project stops at bootstrap inference. Consequently, the reported 95% intervals are confidence intervals for this particular analysis; the project does not estimate empirical coverage by repeating the complete data-generation and missingness process.

The regression model is also deliberately simple, with one response variable and three fully observed predictors. Real datasets may contain multiple incomplete variables, nonlinear relationships, interactions, correlated variables and non-normal outcomes.

The MICE procedure is a simplified PMM-based implementation. In this project only \(Y\) is incomplete, so the usual chained-equation setting with several incomplete variables is reduced to a simpler iterative PMM procedure.

Finally, MNAR cannot generally be resolved from the observed data alone without additional assumptions. The MNAR results should therefore be interpreted as a stress test rather than a complete identification strategy.


# 9. Conclusion

This project examined how missing-data handling methods affect regression estimation and inference under three missingness conditions.

The central comparison was made **within each missingness mechanism**: MCAR methods were compared with one another, MAR methods were compared with one another, and MNAR methods were compared with one another.

The simulation showed that the choice of handling method can substantially change the estimated regression relationship. Mean imputation produced the strongest downward distortion of the focal \(\beta_1\) coefficient in the simulated data, while donor-based, stochastic, multiple-imputation, complete-case, regression-imputation and EM approaches produced different patterns of estimates and uncertainty.

The bootstrap stage demonstrated how the same missing-data problem can also lead to different measures of uncertainty depending on the method used. MICE additionally incorporated variation across multiple imputations through Rubin's rules.

The main lesson is that missing-data handling is not simply a preprocessing decision. Once information is missing, the assumptions used to represent that information become part of the statistical analysis itself.
